# Imports

This section imports all required libraries for:
- File handling
- Numerical computation
- Progress tracking
- Memory diagnostics

We keep this minimal and clean to avoid unnecessary dependencies.

In [1]:
# ================================
# Imports
# ===============================

import os                  # file system operations
import re                  # regex for parsing file names
import glob                # file pattern matching
import math                # math utilities
import time                # timing execution
import gc                  # garbage collection

import psutil              # system + memory monitoring
import numpy as np         # numerical operations
from tqdm import tqdm      # progress bars
from collections import defaultdict  # efficient grouping

# Memory Monitoring Utilities

We are working with large datasets (~50K polygons), so memory usage matters.

This section provides helper functions to:
- Track RAM usage of the process
- Monitor system memory availability
- Print readable diagnostics during heavy operations

Useful during:
- Encoding loading
- Index building
- Query execution

In [2]:
# ================================
# Memory Tracking Utilities
# ================================

def get_memory_stats():
    """
    Returns current memory usage stats for:
    - This Python process
    - Overall system memory
    """
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()

    # Process-level memory
    rss_gb = mem_info.rss / (1024 ** 3)   # Resident memory (actual RAM used)
    vms_gb = mem_info.vms / (1024 ** 3)   # Virtual memory

    # System-level memory
    sys_mem = psutil.virtual_memory()
    avail_gb = sys_mem.available / (1024 ** 3)
    total_gb = sys_mem.total / (1024 ** 3)
    used_pct = sys_mem.percent

    return {
        "rss_gb": rss_gb,
        "vms_gb": vms_gb,
        "avail_gb": avail_gb,
        "total_gb": total_gb,
        "used_pct": used_pct
    }


def print_memory_stats(tag=""):
    """
    Pretty-print memory usage.
    
    Example:
        print_memory_stats("After loading encodings")
    """
    m = get_memory_stats()
    prefix = f"[{tag}] " if tag else ""

    print(
        f"{prefix}"
        f"RSS={m['rss_gb']:.2f} GB | "
        f"VMS={m['vms_gb']:.2f} GB | "
        f"Avail={m['avail_gb']:.2f}/{m['total_gb']:.2f} GB | "
        f"Used={m['used_pct']:.1f}%"
    )

# Load Weighted Encodings (Parallel)

This section loads the weighted encoding files in parallel.

Design:
- each worker parses one encoding file
- file name provides the starting polygon ID
- rows are converted into sparse in-memory form:
  - `ids`
  - `vals`
- the main process merges all parsed rows into the final `encodings` list

This replaces the earlier serial encoding-load step.

In [3]:
# ================================
# Load weighted encodings (parallel) with disk cache
# ================================

import math
import os
import re
import glob
import time
import pickle
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

# ================================
# Parameters
# ================================

# Directory containing weighted encoding files like:
#   real_0.txt, real_250.txt, ...
ENCODING_DIR = "/mnt/data1/ruban/encodings/pk-real0.002"

# File prefix for weighted encoding files
ENCODING_FILE_PREFIX = "real_"

# Ground truth directory
GROUND_TRUTH_DIR = "/mnt/data1/ruban/groundtruth/pk-query-187019"

_dataset_key = os.path.basename(ENCODING_DIR.rstrip("/"))
# Cache directory / file
CACHE_DIR = f"/mnt/data1/ruban/polygon-ann-multi-index/boi_cache/{_dataset_key}/"

os.makedirs(CACHE_DIR, exist_ok=True)
print(f"CACHE_DIR: {CACHE_DIR}")

ENCODINGS_CACHE_PATH = os.path.join(CACHE_DIR, "encodings_payload.pkl")

# Total number of polygons in this dataset
def count_polygons(encoding_dir, file_prefix="real_"):
    pattern = os.path.join(encoding_dir, f"{file_prefix}*.txt")
    files = glob.glob(pattern)
    total = 0
    for fpath in sorted(files):
        with open(fpath, "r") as f:
            total += sum(1 for line in f if line.strip())
    return total

POLY_COUNT = count_polygons(ENCODING_DIR, ENCODING_FILE_PREFIX)
print(f"Auto-detected POLY_COUNT: {POLY_COUNT:,}")

# Split:
# first 80% -> database
# last 20%  -> query set
TRAIN_RATIO = 0.80

# Use full query split for now
MAX_QUERIES = None

# Values with abs(value) <= ZERO_TOL are treated as zero
ZERO_TOL = 1e-12

# In-memory dtypes
ID_DTYPE = np.int32
VAL_DTYPE = np.float32

# Recall points to evaluate later
RECALL_KS = [10, 50, 500]

# Verbose diagnostics
VERBOSE = True

# Derived split indices
DATA_END = math.ceil(POLY_COUNT * TRAIN_RATIO)
QUERY_START = DATA_END
QUERY_END = POLY_COUNT if MAX_QUERIES is None else min(POLY_COUNT, QUERY_START + MAX_QUERIES)

print("=== Parameter Summary ===")
print(f"ENCODING_DIR           : {ENCODING_DIR}")
print(f"ENCODING_FILE_PREFIX   : {ENCODING_FILE_PREFIX}")
print(f"GROUND_TRUTH_DIR       : {GROUND_TRUTH_DIR}")
print(f"POLY_COUNT             : {POLY_COUNT:,}")
print(f"TRAIN_RATIO            : {TRAIN_RATIO}")
print(f"DATA_END               : {DATA_END:,}")
print(f"QUERY_START            : {QUERY_START:,}")
print(f"QUERY_END              : {QUERY_END:,}")
print(f"NUM_QUERIES            : {QUERY_END - QUERY_START:,}")
print(f"ZERO_TOL               : {ZERO_TOL}")
print(f"ID_DTYPE               : {ID_DTYPE}")
print(f"VAL_DTYPE              : {VAL_DTYPE}")
print(f"RECALL_KS              : {RECALL_KS}")
print(f"CACHE_DIR              : {CACHE_DIR}")
print(f"ENCODINGS_CACHE_PATH   : {ENCODINGS_CACHE_PATH}")

print_memory_stats("After parameters")

# Parallel settings
ENC_LOAD_WORKERS = 16

print("=== Parallel Encoding Load Parameters ===")
print(f"ENC_LOAD_WORKERS       : {ENC_LOAD_WORKERS}")
print(f"ENCODING_DIR           : {ENCODING_DIR}")
print(f"ENCODING_FILE_PREFIX   : {ENCODING_FILE_PREFIX}")


def _parse_weighted_line_auto_worker(line, zero_tol=1e-12):
    """
    Worker-safe line parser.

    Supported formats:
    1) Dense floats
    2) Sparse idx:value
    3) Sparse alternating idx value pairs
    """
    line = line.strip()

    if not line:
        return (
            np.empty(0, dtype=ID_DTYPE),
            np.empty(0, dtype=VAL_DTYPE)
        )

    toks = line.split()

    # Sparse idx:value
    if ":" in toks[0]:
        ids = []
        vals = []

        for tok in toks:
            cell_str, val_str = tok.split(":", 1)
            v = float(val_str)
            if abs(v) > zero_tol:
                ids.append(int(cell_str))
                vals.append(v)

        return (
            np.asarray(ids, dtype=ID_DTYPE),
            np.asarray(vals, dtype=VAL_DTYPE)
        )

    # Sparse alternating idx value
    is_even = (len(toks) % 2 == 0)

    def _looks_like_int_token(s):
        return re.fullmatch(r"[+-]?\d+", s) is not None

    if is_even and all(_looks_like_int_token(toks[i]) for i in range(0, len(toks), 2)):
        ids = []
        vals = []

        for i in range(0, len(toks), 2):
            cell_id = int(toks[i])
            v = float(toks[i + 1])

            if abs(v) > zero_tol:
                ids.append(cell_id)
                vals.append(v)

        return (
            np.asarray(ids, dtype=ID_DTYPE),
            np.asarray(vals, dtype=VAL_DTYPE)
        )

    # Dense float row
    dense = np.fromstring(line, sep=" ", dtype=VAL_DTYPE)

    if dense.size == 0:
        return (
            np.empty(0, dtype=ID_DTYPE),
            np.empty(0, dtype=VAL_DTYPE)
        )

    nz = np.flatnonzero(np.abs(dense) > zero_tol).astype(ID_DTYPE)
    vals = dense[nz].astype(VAL_DTYPE, copy=False)

    return nz, vals


def _extract_file_start_id(fpath, file_prefix):
    base = os.path.basename(fpath)
    m = re.search(rf"{re.escape(file_prefix)}(\d+)\.txt$", base)
    if not m:
        raise ValueError(f"Unexpected encoding file name format: {base}")
    return int(m.group(1))


def _parse_one_encoding_file(args):
    """
    Worker function for one encoding file.

    Returns
    -------
    rows : list of (poly_id, ids, vals)
    """
    fpath, file_prefix, start_id, end_id, zero_tol = args

    file_start_id = _extract_file_start_id(fpath, file_prefix)
    rows = []

    with open(fpath, "r") as f:
        for local_row_idx, line in enumerate(f):
            poly_id = file_start_id + local_row_idx

            if poly_id < start_id or poly_id >= end_id:
                continue

            ids, vals = _parse_weighted_line_auto_worker(line, zero_tol=zero_tol)
            rows.append((poly_id, ids, vals))

    return rows


def load_weighted_encodings_parallel(encoding_dir, start_id=0, end_id=None,
                                     file_prefix="real_", zero_tol=1e-12,
                                     num_workers=16, show_progress=True):
    """
    Parallel weighted encoding loader.

    Returns
    -------
    encodings : list[(ids, vals)]
    """
    if end_id is None:
        end_id = POLY_COUNT

    pattern = os.path.join(encoding_dir, f"{file_prefix}*.txt")
    files = glob.glob(pattern)

    if not files:
        raise FileNotFoundError(f"No files found matching pattern: {pattern}")

    files = sorted(files, key=lambda f: _extract_file_start_id(f, file_prefix))

    n_rows = end_id - start_id
    empty_ids = np.empty(0, dtype=ID_DTYPE)
    empty_vals = np.empty(0, dtype=VAL_DTYPE)
    encodings = [(empty_ids, empty_vals) for _ in range(n_rows)]

    jobs = [
        (fpath, file_prefix, start_id, end_id, zero_tol)
        for fpath in files
    ]

    ctx = mp.get_context("fork")

    loaded_rows = 0
    nonempty_rows = 0

    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as ex:
        futures = [ex.submit(_parse_one_encoding_file, job) for job in jobs]

        iterator = as_completed(futures)
        if show_progress:
            iterator = tqdm(iterator, total=len(futures), desc="Loading weighted encodings (parallel)", unit="file")

        for fut in iterator:
            rows = fut.result()

            for poly_id, ids, vals in rows:
                encodings[poly_id - start_id] = (ids, vals)
                loaded_rows += 1
                if ids.size > 0:
                    nonempty_rows += 1

    print(f"Loaded rows             : {loaded_rows:,}")
    print(f"Non-empty rows          : {nonempty_rows:,}")
    print(f"Requested poly ID range : [{start_id}, {end_id})")

    return encodings


# ================================
# Cache-aware load
# ================================
if os.path.exists(ENCODINGS_CACHE_PATH):   
    print(f"[Cache hit] Loading encodings from disk: {ENCODINGS_CACHE_PATH}")
    t0 = time.time()

    with open(ENCODINGS_CACHE_PATH, "rb") as f:
        enc_payload = pickle.load(f)

    encodings = enc_payload["encodings"]

    # restore if present
    if "DATA_END" in enc_payload:
        DATA_END = enc_payload["DATA_END"]
    if "QUERY_START" in enc_payload:
        QUERY_START = enc_payload["QUERY_START"]
    if "QUERY_END" in enc_payload:
        QUERY_END = enc_payload["QUERY_END"]
    if "POLY_COUNT" in enc_payload:
        POLY_COUNT = enc_payload["POLY_COUNT"]

    t_load_enc = time.time() - t0
    print(f"Loaded cached encodings in {t_load_enc:.2f} sec")

else:
    print("[Cache miss] Loading weighted encodings fresh...")
    t0 = time.time()

    encodings = load_weighted_encodings_parallel(
        encoding_dir=ENCODING_DIR,
        start_id=0,
        end_id=POLY_COUNT,
        file_prefix=ENCODING_FILE_PREFIX,
        zero_tol=ZERO_TOL,
        num_workers=ENC_LOAD_WORKERS,
        show_progress=True
    )

    t_load_enc = time.time() - t0
    print(f"\nFresh parallel encoding load time: {t_load_enc:.2f} sec ({t_load_enc / 60:.2f} min)")

    print(f"[Cache save] Writing encodings to: {ENCODINGS_CACHE_PATH}")
    with open(ENCODINGS_CACHE_PATH, "wb") as f:
        pickle.dump({
            "encodings": encodings,
            "POLY_COUNT": POLY_COUNT,
            "DATA_END": DATA_END,
            "QUERY_START": QUERY_START,
            "QUERY_END": QUERY_END,
            "ZERO_TOL": ZERO_TOL,
            "ID_DTYPE": str(ID_DTYPE),
            "VAL_DTYPE": str(VAL_DTYPE),
        }, f, protocol=pickle.HIGHEST_PROTOCOL)

print_memory_stats("After encoding load")

# ================================
# Basic weighted encoding diagnostics
# ================================
nnz_counts = np.array([ids.size for ids, vals in encodings], dtype=np.int32)

nonempty_mask = (nnz_counts > 0)
num_nonempty = int(nonempty_mask.sum())
num_empty = int((~nonempty_mask).sum())

max_cell_id = -1
min_nonzero_val = None
max_nonzero_val = None

for ids, vals in encodings:
    if ids.size > 0:
        local_max_id = int(ids.max())
        if local_max_id > max_cell_id:
            max_cell_id = local_max_id

    if vals.size > 0:
        local_min_val = float(vals.min())
        local_max_val = float(vals.max())

        if min_nonzero_val is None or local_min_val < min_nonzero_val:
            min_nonzero_val = local_min_val

        if max_nonzero_val is None or local_max_val > max_nonzero_val:
            max_nonzero_val = local_max_val

print("\n=== Weighted Encoding Diagnostics ===")
print(f"Total polygons           : {len(encodings):,}")
print(f"Non-empty polygons       : {num_nonempty:,}")
print(f"Empty polygons           : {num_empty:,}")
print(f"Mean nnz / polygon       : {nnz_counts.mean():.2f}")
print(f"Median nnz / polygon     : {np.median(nnz_counts):.2f}")
print(f"p90 nnz / polygon        : {np.percentile(nnz_counts, 90):.2f}")
print(f"p99 nnz / polygon        : {np.percentile(nnz_counts, 99):.2f}")
print(f"Max nnz / polygon        : {nnz_counts.max()}")
print(f"Max cell ID seen         : {max_cell_id}")
print(f"Min nonzero value        : {min_nonzero_val}")
print(f"Max nonzero value        : {max_nonzero_val}")

CACHE_DIR: /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/
Auto-detected POLY_COUNT: 233,773
=== Parameter Summary ===
ENCODING_DIR           : /mnt/data1/ruban/encodings/pk-real0.002
ENCODING_FILE_PREFIX   : real_
GROUND_TRUTH_DIR       : /mnt/data1/ruban/groundtruth/pk-query-187019
POLY_COUNT             : 233,773
TRAIN_RATIO            : 0.8
DATA_END               : 187,019
QUERY_START            : 187,019
QUERY_END              : 233,773
NUM_QUERIES            : 46,754
ZERO_TOL               : 1e-12
ID_DTYPE               : <class 'numpy.int32'>
VAL_DTYPE              : <class 'numpy.float32'>
RECALL_KS              : [10, 50, 500]
CACHE_DIR              : /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/
ENCODINGS_CACHE_PATH   : /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/encodings_payload.pkl
[After parameters] RSS=0.07 GB | VMS=2.64 GB | Avail=57.65/61.95 GB | Used=6.9%
=== Parallel Encoding Load Parameters ===
ENC_LOAD_WORKERS

# Ground Truth Loader

This section loads the existing polygon-shape ground truth.

Format assumed from the current notebook:
- one query per line
- each line is:
  `query_id, neighbor1, neighbor2, neighbor3, ...`

This is the same exact-polygon Jaccard based ground truth used in the earlier pipeline.

We will use it only for evaluation and diagnostics.

In [4]:
# ================================
# Ground truth loader with disk cache
# ================================

import os
import time
import pickle
import numpy as np

GT_CACHE_PATH = os.path.join(CACHE_DIR, "gt.pkl")

def read_ground_truth(path):
    """
    Load ground truth files and return a CSR-style structure.

    Returns
    -------
    gt_neighbors : np.ndarray[int32]   — flat array of all neighbor IDs
    gt_offsets   : np.ndarray[int64]   — gt_offsets[i]..gt_offsets[i+1] = neighbors of query i
    gt_qids      : np.ndarray[int32]   — sorted query IDs
    gt_qid_map   : dict[int, int]      — maps query ID -> row index into gt_offsets
    """
    raw = {}
    files = sorted(os.listdir(path))

    for fname in files:
        fpath = os.path.join(path, fname)
        if not os.path.isfile(fpath):
            continue
        with open(fpath, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split(", ")
                if len(parts) >= 2:
                    qid = int(parts[0])
                    neighbors = [int(x) for x in parts[1:] if x.strip()]
                    raw[qid] = neighbors

    # Build CSR layout
    gt_qids = np.array(sorted(raw.keys()), dtype=np.int32)
    gt_offsets = np.zeros(len(gt_qids) + 1, dtype=np.int64)

    for i, qid in enumerate(gt_qids):
        gt_offsets[i + 1] = gt_offsets[i] + len(raw[qid])

    total = int(gt_offsets[-1])
    gt_neighbors = np.empty(total, dtype=np.int32)

    for i, qid in enumerate(gt_qids):
        s, e = gt_offsets[i], gt_offsets[i + 1]
        gt_neighbors[s:e] = raw[qid]

    gt_qid_map = {int(qid): i for i, qid in enumerate(gt_qids)}

    print(f"Ground truth loaded: {len(gt_qids):,} queries | {total:,} total neighbors")
    return gt_neighbors, gt_offsets, gt_qids, gt_qid_map


# -------------------------
# Load GT (cache-aware)
# -------------------------

print("[2] Loading ground truth...")
t0 = time.time()

GT_CSR_CACHE_PATH = os.path.join(CACHE_DIR, "gt_csr.pkl")
if os.path.exists(GT_CSR_CACHE_PATH):
    print(f"[Cache hit] Loading GT CSR from disk: {GT_CSR_CACHE_PATH}")
    with open(GT_CSR_CACHE_PATH, "rb") as f:
        _gt_payload = pickle.load(f)
    gt_neighbors = _gt_payload["gt_neighbors"]
    gt_offsets   = _gt_payload["gt_offsets"]
    gt_qids      = _gt_payload["gt_qids"]
    gt_qid_map   = _gt_payload["gt_qid_map"]
else:
    print(f"[Cache miss] Loading GT fresh from: {GROUND_TRUTH_DIR}")
    gt_neighbors, gt_offsets, gt_qids, gt_qid_map = read_ground_truth(GROUND_TRUTH_DIR)

    print(f"[Cache save] Writing GT CSR to: {GT_CSR_CACHE_PATH}")
    with open(GT_CSR_CACHE_PATH, "wb") as f:
        pickle.dump({
            "gt_neighbors": gt_neighbors,
            "gt_offsets":   gt_offsets,
            "gt_qids":      gt_qids,
            "gt_qid_map":   gt_qid_map,
        }, f, protocol=pickle.HIGHEST_PROTOCOL)

t_gt = time.time() - t0

print(f"Ground truth load time: {t_gt:.2f} sec")
print_memory_stats("After loading ground truth")

# -------------------------
# GT diagnostics
# -------------------------

gt_lengths = (gt_offsets[1:] - gt_offsets[:-1]).astype(np.int32)

print("\n=== Ground Truth Diagnostics ===")
print(f"GT query count            : {len(gt_qids):,}")
print(f"Min GT query ID           : {gt_qids[0]}")
print(f"Max GT query ID           : {gt_qids[-1]}")
print(f"Mean GT list length       : {gt_lengths.mean():.2f}")
print(f"Median GT list length     : {np.median(gt_lengths):.2f}")
print(f"Min GT list length        : {gt_lengths.min()}")
print(f"Max GT list length        : {gt_lengths.max()}")
print(f"p90 GT list length        : {np.percentile(gt_lengths, 90):.2f}")

query_ids = list(range(QUERY_START, QUERY_END))
valid_gt_query_ids = [qid for qid in query_ids if qid in gt_qid_map]

print("\n=== Query Split vs GT ===")
print(f"Query split size          : {len(query_ids):,}")
print(f"Queries with GT           : {len(valid_gt_query_ids):,}")
print(f"Queries without GT        : {len(query_ids) - len(valid_gt_query_ids):,}")

[2] Loading ground truth...
[Cache hit] Loading GT CSR from disk: /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/gt_csr.pkl
Ground truth load time: 0.49 sec
[After loading ground truth] RSS=9.81 GB | VMS=12.37 GB | Avail=47.47/61.95 GB | Used=23.4%

=== Ground Truth Diagnostics ===
GT query count            : 44,666
Min GT query ID           : 187019
Max GT query ID           : 233772
Mean GT list length       : 4843.89
Median GT list length     : 3275.00
Min GT list length        : 1
Max GT list length        : 17410
p90 GT list length        : 12445.00

=== Query Split vs GT ===
Query split size          : 46,754
Queries with GT           : 44,666
Queries without GT        : 2,088


# Weighted MinHash Signatures

Now that weighted Jaccard is verified, we replace standard MinHash with weighted MinHash.

Goal:
- create compact signatures for weighted vectors
- preserve weighted Jaccard similarity approximately
- keep the rest of the banding pipeline structurally similar to the earlier notebook

Implementation choice:
- use Consistent Weighted Sampling (CWS), specifically the improved weighted MinHash method
- each signature position produces one integer-like bucket value
- the final signature matrix will later be used for banding

Important:
- this is only the signature-generation stage
- we are not building buckets or indexes yet

In [5]:
# ================================
# Weighted MinHash setup
# ================================

# Signature length
# Keep this aligned with your earlier pipeline for easier comparison
L = 100

# Random seed for reproducibility
WMH_SEED = 42

print("=== Weighted MinHash Parameters ===")
print(f"L         : {L}")
print(f"WMH_SEED  : {WMH_SEED}")


def build_weighted_minhash_params(L, seed=42):
    """
    Build random parameters for Improved Consistent Weighted Sampling (ICWS-style).

    For each signature dimension j, and each active feature i, we use:
      r_ij ~ Gamma(2,1)
      c_ij ~ Gamma(2,1)
      beta_ij ~ Uniform(0,1)

    In practice, we generate these lazily per active coordinate by indexing
    pre-generated matrices of size [L, D], where D is the max cell id + 1.

    Since D is known only after loading the data, we build these params after
    encoding diagnostics.

    Returns
    -------
    params : dict
        Contains random parameter matrices needed for weighted MinHash
    """
    D = max(int(ids.max()) for ids, vals in encodings if ids.size > 0) + 1
    rng = np.random.default_rng(seed)

    # Gamma(shape=2, scale=1) and Uniform(0,1)
    r = rng.gamma(shape=2.0, scale=1.0, size=(L, D)).astype(np.float32)
    c = rng.gamma(shape=2.0, scale=1.0, size=(L, D)).astype(np.float32)
    beta = rng.random(size=(L, D), dtype=np.float32)

    return {
        "L": L,
        "D": D,
        "r": r,
        "c": c,
        "beta": beta
    }


def weighted_minhash_signature_one(ids, vals, params):
    """
    Compute one weighted MinHash signature for a sparse nonnegative vector.

    Parameters
    ----------
    ids : np.ndarray[int]
        Active feature IDs
    vals : np.ndarray[float]
        Positive weights for those features
    params : dict
        Random parameters produced by build_weighted_minhash_params()

    Returns
    -------
    sig : np.ndarray[int64] of shape (L,)
        Weighted MinHash signature
    """
    L = params["L"]
    r = params["r"]
    c = params["c"]
    beta = params["beta"]

    # Empty vector: assign a sentinel signature
    if ids.size == 0:
        return np.full(L, -1, dtype=np.int64)

    # Safety: weighted MinHash requires positive weights
    positive_mask = vals > 0
    ids = ids[positive_mask]
    vals = vals[positive_mask]

    if ids.size == 0:
        return np.full(L, -1, dtype=np.int64)

    log_vals = np.log(vals.astype(np.float32))

    sig = np.empty(L, dtype=np.int64)

    # For each signature dimension, choose the feature with minimum a_z
    for j in range(L):
        r_j = r[j, ids]
        c_j = c[j, ids]
        beta_j = beta[j, ids]

        # ICWS-style transform
        t = np.floor((log_vals / r_j) + beta_j)
        y = np.exp(r_j * (t - beta_j))
        a = c_j / (y * np.exp(r_j))

        # Winner feature index within current active set
        k = int(np.argmin(a))

        # Encode both feature id and discretized t into one hashable integer
        # This preserves more information than feature id alone.
        feature_id = int(ids[k])
        t_val = int(t[k])

        sig[j] = np.int64(feature_id) * np.int64(2147483647) + np.int64(t_val)

    return sig


print("Building weighted MinHash random parameters...")
t0 = time.time()
wmh_params = build_weighted_minhash_params(L=L, seed=WMH_SEED)
t_build_wmh = time.time() - t0

print(f"Weighted MinHash parameter build time: {t_build_wmh:.2f} sec")
print(f"Feature dimension D                  : {wmh_params['D']:,}")
print_memory_stats("After WMH parameter build")

=== Weighted MinHash Parameters ===
L         : 100
WMH_SEED  : 42
Building weighted MinHash random parameters...
Weighted MinHash parameter build time: 0.60 sec
Feature dimension D                  : 18,219
[After WMH parameter build] RSS=9.84 GB | VMS=12.40 GB | Avail=47.48/61.95 GB | Used=23.4%


# Generate Weighted MinHash Signatures (Parallel)

This section computes weighted MinHash signatures in parallel.

Design:
- split the polygon encodings into chunks
- use `ProcessPoolExecutor`
- each worker computes signatures for one chunk
- merge chunk results into the final signature matrix

This replaces the earlier serial signature-build step.

In [6]:
# ================================
# Generate weighted MinHash signatures (parallel) with disk cache
# ================================

import os
import time
import pickle
import numpy as np
from numba import njit, prange

# ------------------------------------------------------------
# Cache paths
# ------------------------------------------------------------
# Assumes CACHE_DIR already exists
FLAT_ENCODING_CACHE_PATH = os.path.join(CACHE_DIR, "flat_encoding_arrays.pkl")
WMH_PARAMS_CACHE_PATH = os.path.join(CACHE_DIR, f"wmh_params_L{L}.pkl")
SIG_ALL_CACHE_PATH = os.path.join(CACHE_DIR, f"sig_all_L{L}.npy")

print("=== WMH Cache Paths ===")
print(f"FLAT_ENCODING_CACHE_PATH : {FLAT_ENCODING_CACHE_PATH}")
print(f"WMH_PARAMS_CACHE_PATH    : {WMH_PARAMS_CACHE_PATH}")
print(f"SIG_ALL_CACHE_PATH       : {SIG_ALL_CACHE_PATH}")

# "hnsw"     → existing nmslib WeightedJaccard HNSW (baseline)
# "nested_lsh" → second-level MinHash banding inside large buckets
# "ivf_flat"   → FAISS IVFFlat on 100-dim signatures
INDEX_TYPE = "nested_lsh"

@njit(cache=True, parallel=True)
def _wmh_sig_all_numba(ids_list_flat, vals_list_flat, offsets, r, c, beta, L, N):
    """
    Compute all N signatures in parallel using Numba prange.
    ids_list_flat / vals_list_flat are 1D concatenations of all sparse vectors.
    offsets[i] = start index of polygon i in the flat arrays.
    offsets[N] = total length (sentinel).
    """
    sig_all = np.full((N, L), -1, dtype=np.int64)

    for poly_idx in prange(N):
        start = offsets[poly_idx]
        end   = offsets[poly_idx + 1]

        if start == end:
            continue

        ids  = ids_list_flat[start:end]
        vals = vals_list_flat[start:end].copy()

        n = 0
        for ki in range(ids.shape[0]):
            if vals[ki] > 0.0:
                vals[n] = np.log(vals[ki])
                ids[n]  = ids[ki]
                n += 1

        if n == 0:
            continue

        ids  = ids[:n]
        vals = vals[:n]

        for j in range(L):
            best_a          = np.inf
            best_feature_id = np.int64(-1)
            best_t          = np.int64(0)

            for ki in range(n):
                idx    = np.int64(ids[ki])
                r_j    = r[j, idx]
                c_j    = c[j, idx]
                beta_j = beta[j, idx]
                lv     = vals[ki]

                t = np.floor(lv / r_j + beta_j)
                y = np.exp(r_j * (t - beta_j))
                a = c_j / (y * np.exp(r_j))

                if a < best_a:
                    best_a          = a
                    best_feature_id = idx
                    best_t          = np.int64(t)

            sig_all[poly_idx, j] = best_feature_id * np.int64(2147483647) + best_t

    return sig_all


# ------------------------------------------------------------
# 1) Flat encoding arrays (cache-aware)
# ------------------------------------------------------------
if os.path.exists(FLAT_ENCODING_CACHE_PATH):
    print(f"[Cache hit] Loading flat encoding arrays from: {FLAT_ENCODING_CACHE_PATH}")
    t_flat0 = time.time()

    with open(FLAT_ENCODING_CACHE_PATH, "rb") as f:
        flat_payload = pickle.load(f)

    ids_flat = flat_payload["ids_flat"]
    vals_flat = flat_payload["vals_flat"]
    offsets = flat_payload["offsets"]
    total_nnz = flat_payload["total_nnz"]

    # Safety check — ensure correct dtypes
    assert ids_flat.dtype == np.int32, f"ids_flat dtype mismatch: expected int32, got {ids_flat.dtype}"
    assert vals_flat.dtype == np.float32, f"vals_flat dtype mismatch: expected float32, got {vals_flat.dtype}"
    print(f"ids_flat dtype  : {ids_flat.dtype} ✓")
    print(f"vals_flat dtype : {vals_flat.dtype} ✓")

    print(f"Loaded flat arrays in {time.time() - t_flat0:.2f} sec | total nnz: {total_nnz:,}")

else:
    print("[Cache miss] Building flat encoding arrays for Numba...")
    t_flat0 = time.time()

    offsets = np.zeros(len(encodings) + 1, dtype=np.int64)
    total_nnz = 0

    for i, (ids, vals) in enumerate(encodings):
        total_nnz += ids.size
        offsets[i + 1] = total_nnz

    ids_flat = np.empty(total_nnz, dtype=np.int32)
    vals_flat = np.empty(total_nnz, dtype=np.float32)

    for i, (ids, vals) in enumerate(encodings):
        s, e = offsets[i], offsets[i + 1]
        ids_flat[s:e] = ids
        vals_flat[s:e] = vals

    print(f"Flat array build: {time.time() - t_flat0:.2f} sec | total nnz: {total_nnz:,}")

    print(f"[Cache save] Writing flat encoding arrays to: {FLAT_ENCODING_CACHE_PATH}")
    with open(FLAT_ENCODING_CACHE_PATH, "wb") as f:
        pickle.dump({
            "ids_flat": ids_flat,
            "vals_flat": vals_flat,
            "offsets": offsets,
            "total_nnz": total_nnz,
        }, f, protocol=pickle.HIGHEST_PROTOCOL)


# ------------------------------------------------------------
# 2) WMH params for current L (cache-aware)
# ------------------------------------------------------------
if os.path.exists(WMH_PARAMS_CACHE_PATH):
    print(f"[Cache hit] Loading WMH params from: {WMH_PARAMS_CACHE_PATH}")
    t_params0 = time.time()

    with open(WMH_PARAMS_CACHE_PATH, "rb") as f:
        wmh_params = pickle.load(f)

    print(f"Loaded WMH params in {time.time() - t_params0:.2f} sec")

else:
    print(f"[Cache miss] Building WMH params for L={L} ...")
    t_params0 = time.time()

    wmh_params = build_weighted_minhash_params(L, seed=WMH_SEED)

    print(f"WMH params build time: {time.time() - t_params0:.2f} sec")

    print(f"[Cache save] Writing WMH params to: {WMH_PARAMS_CACHE_PATH}")
    with open(WMH_PARAMS_CACHE_PATH, "wb") as f:
        pickle.dump(wmh_params, f, protocol=pickle.HIGHEST_PROTOCOL)


# ------------------------------------------------------------
# 3) Numba warmup (only if needed)
# ------------------------------------------------------------
print("Numba WMH warmup...")
_ = _wmh_sig_all_numba(
    ids_flat[:offsets[1]],
    vals_flat[:offsets[1]],
    np.array([0, offsets[1]], dtype=np.int64),
    wmh_params["r"], wmh_params["c"], wmh_params["beta"],
    L, 1
)
print("Numba WMH signature warmup done — compiled and ready.")


# ------------------------------------------------------------
# 4) Signature matrix for current L (cache-aware)
# ------------------------------------------------------------
if os.path.exists(SIG_ALL_CACHE_PATH):
    print(f"[Cache hit] Loading weighted MinHash signatures from: {SIG_ALL_CACHE_PATH}")
    t0 = time.time()

    sig_all = np.load(SIG_ALL_CACHE_PATH)
    print(f"sig_all raw dtype: {sig_all.dtype}")
    if sig_all.dtype != np.int64:
        print(f"Converting sig_all from {sig_all.dtype} to int64...")
        sig_all = sig_all.astype(np.int64)
        np.save(SIG_ALL_CACHE_PATH, sig_all)
        print("Cache updated.")

    t_sig = time.time() - t0
    print(f"Loaded cached signatures in {t_sig:.2f} sec")

else:
    print("[Cache miss] Computing weighted MinHash signatures...")
    t0 = time.time()

    sig_all = _wmh_sig_all_numba(
        ids_flat, vals_flat, offsets,
        wmh_params["r"], wmh_params["c"], wmh_params["beta"],
        L, len(encodings)
    )

    t_sig = time.time() - t0
    print(f"\nNumba parallel weighted MinHash signature time: {t_sig:.2f} sec ({t_sig/60:.2f} min)")

    print(f"[Cache save] Writing signatures to: {SIG_ALL_CACHE_PATH}")
    np.save(SIG_ALL_CACHE_PATH, sig_all.astype(np.float32))


print(f"Signature matrix shape                        : {sig_all.shape}")
print(f"Signature dtype                               : {sig_all.dtype}")


print_memory_stats("After weighted MinHash signatures")

=== WMH Cache Paths ===
FLAT_ENCODING_CACHE_PATH : /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/flat_encoding_arrays.pkl
WMH_PARAMS_CACHE_PATH    : /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/wmh_params_L100.pkl
SIG_ALL_CACHE_PATH       : /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/sig_all_L100.npy
[Cache hit] Loading flat encoding arrays from: /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/flat_encoding_arrays.pkl
ids_flat dtype  : int32 ✓
vals_flat dtype : float32 ✓
Loaded flat arrays in 4.95 sec | total nnz: 1,147,422,841
[Cache hit] Loading WMH params from: /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/wmh_params_L100.pkl
Loaded WMH params in 0.01 sec
Numba WMH warmup...
Numba WMH signature warmup done — compiled and ready.
[Cache hit] Loading weighted MinHash signatures from: /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/sig_all_L100.npy
sig_all raw dtype: int64
Loaded c

# Banding Setup

Now we split each weighted MinHash signature into bands.

Purpose:
- generate bucket keys from contiguous chunks of the signature
- group polygons that collide in at least one band
- preserve the overall structure of the earlier BOI pipeline

Definitions:
- `L` = signature length
- `B` = number of bands
- `R` = rows per band
- must satisfy: `L = B * R`

For now, we keep the same style as the earlier notebook so comparisons remain fair.

In [7]:
# ================================
# Banding configuration
# ================================

# Keep this aligned with the earlier pipeline for easier comparison
B = 50
R = 2

assert L == B * R, f"L must equal B * R, but got L={L}, B={B}, R={R}"

print("=== Banding Parameters ===")
print(f"L : {L}")
print(f"B : {B}")
print(f"R : {R}")


# ================================
# Banding helper
# ================================

class Banding:
    """
    Simple helper to split signatures into B bands of size R.
    """

    def __init__(self, B, R):
        self.B = B
        self.R = R
        self.L = B * R

    def get_band(self, sig, band_idx):
        """
        Return one band as a tuple of R signature values.
        """
        start = band_idx * self.R
        end = start + self.R
        return tuple(sig[start:end])

    def get_all_bands(self, sig):
        """
        Return all bands for one signature.
        """
        return [self.get_band(sig, b) for b in range(self.B)]


banding = Banding(B=B, R=R)


# ================================
# Split signature matrix
# ================================

sig_data = sig_all[:DATA_END]
sig_query = sig_all[QUERY_START:QUERY_END]

print("\n=== Signature Split Diagnostics ===")
print(f"sig_data shape   : {sig_data.shape}")
print(f"sig_query shape  : {sig_query.shape}")

print_memory_stats("After banding setup")

=== Banding Parameters ===
L : 100
B : 50
R : 2

=== Signature Split Diagnostics ===
sig_data shape   : (187019, 100)
sig_query shape  : (46754, 100)
[After banding setup] RSS=18.66 GB | VMS=24.19 GB | Avail=39.27/61.95 GB | Used=36.6%


In [8]:
from collections import defaultdict
import numpy as np

def _secondary_split_members(members, sig_data, band_idx, R, max_bucket_size):
    """
    Split a large bucket using the next R signature values as a secondary key.
    sig_data is DB-only signatures.
    """
    if len(members) <= max_bucket_size:
        return [members]

    L = sig_data.shape[1]
    sec_start = ((band_idx + 1) * R) % max(1, L - R + 1)
    sec_end = min(sec_start + R, L)

    groups = defaultdict(list)
    for db_pid in members:
        sec_key = tuple(sig_data[db_pid, sec_start:sec_end].tolist())
        groups[sec_key].append(db_pid)

    out = list(groups.values())

    # fallback if split is useless
    if len(out) == 1:
        out = []
        for i in range(0, len(members), max_bucket_size):
            out.append(members[i:i + max_bucket_size])

    return out


def rebalance_part_map(part_map, sig_data, R, merge_small_threshold=20, split_large_threshold=5000):
    """
    1. Merge tiny buckets into per-band overflow bucket
    2. Split huge buckets using secondary signature slice
    """
    merged_map = {}
    overflow_by_band = defaultdict(list)

    # merge small buckets
    for (band_idx, band_key), members in part_map.items():
        members = list(members)
        if merge_small_threshold > 0 and len(members) < merge_small_threshold:
            overflow_by_band[band_idx].extend(members)
        else:
            merged_map[(band_idx, band_key)] = members

    for band_idx, overflow_members in overflow_by_band.items():
        if overflow_members:
            uniq = sorted(set(overflow_members))
            merged_map[(band_idx, "__OVERFLOW__")] = uniq

    # split large buckets
    final_map = {}
    for (band_idx, band_key), members in merged_map.items():
        members = sorted(set(members))

        if band_key != "__OVERFLOW__" and len(members) > split_large_threshold:
            sub_buckets = _secondary_split_members(
                members=members,
                sig_data=sig_data,
                band_idx=band_idx,
                R=R,
                max_bucket_size=split_large_threshold
            )
            for j, sub_members in enumerate(sub_buckets):
                final_map[(band_idx, band_key, "SPLIT", j)] = np.asarray(sorted(set(sub_members)), dtype=np.int32)
        else:
            final_map[(band_idx, band_key)] = np.asarray(members, dtype=np.int32)

    stats = {}
    sizes = np.array([len(v) for v in final_map.values()], dtype=np.int32)
    stats["total_buckets"] = len(final_map)
    stats["mean_bucket_size"] = float(sizes.mean()) if len(sizes) else 0.0
    stats["median_bucket_size"] = float(np.median(sizes)) if len(sizes) else 0.0
    stats["max_bucket_size"] = int(sizes.max()) if len(sizes) else 0
    stats["p90_bucket_size"] = float(np.percentile(sizes, 90)) if len(sizes) else 0.0
    stats["p99_bucket_size"] = float(np.percentile(sizes, 99)) if len(sizes) else 0.0

    return final_map, stats

# Build Band Partitions

This section builds the band-level partition map for the database polygons.

For each database polygon:
- take its weighted MinHash signature
- split it into `B` bands
- use each band tuple as a bucket key

Output:
- a mapping from `(band_id, band_key)` to the list of database polygon IDs that fall into that bucket

This restores the same broad BOI-style structure as the earlier notebook, but now using weighted MinHash signatures.

In [9]:
# ================================
# Build partition map from database signatures
# ================================

def build_partition_map(sig_data, banding):
    """
    Build the band partition map for database signatures.

    Parameters
    ----------
    sig_data : np.ndarray[int64] of shape (N_db, L)
        Signature matrix for database polygons only
    banding : Banding
        Banding helper object

    Returns
    -------
    part_map : dict
        Maps (band_idx, band_key) -> list of database polygon IDs
    bucket_sizes : list[int]
        Sizes of all buckets
    """
    part_map = defaultdict(list)

    for db_pid in tqdm(range(sig_data.shape[0]), desc="Building band partitions", unit="poly"):
        sig = sig_data[db_pid]

        for band_idx in range(banding.B):
            band_key = banding.get_band(sig, band_idx)
            part_map[(band_idx, band_key)].append(db_pid)

    bucket_sizes = [len(v) for v in part_map.values()]
    return part_map, bucket_sizes


print("[4] Building partition map from weighted MinHash signatures...")
t0 = time.time()

part_map_raw, bucket_sizes_raw = build_partition_map(sig_data=sig_data, banding=banding)

REB_MERGE_SMALL = 0
REB_SPLIT_LARGE = 3000

part_map, rebalance_stats = rebalance_part_map(
    part_map=part_map_raw,
    sig_data=sig_data,
    R=banding.R,
    merge_small_threshold=REB_MERGE_SMALL,
    split_large_threshold=REB_SPLIT_LARGE
)

bucket_sizes = [len(v) for v in part_map.values()]

t_part = time.time() - t0

print(f"\nPartition build time      : {t_part:.2f} sec")
print(f"Total unique buckets      : {len(part_map):,}")
print(f"Total bucket assignments  : {sum(bucket_sizes):,}")

print("\n=== Rebalance Settings ===")
print(f"REB_MERGE_SMALL          : {REB_MERGE_SMALL}")
print(f"REB_SPLIT_LARGE          : {REB_SPLIT_LARGE}")

print("\n=== Rebalanced Bucket Stats ===")
for k, v in rebalance_stats.items():
    print(f"{k:25s}: {v}")

bucket_sizes_arr = np.array(bucket_sizes, dtype=np.int32)

print("\n=== Bucket Size Diagnostics ===")
print(f"Mean bucket size          : {bucket_sizes_arr.mean():.2f}")
print(f"Median bucket size        : {np.median(bucket_sizes_arr):.2f}")
print(f"Min bucket size           : {bucket_sizes_arr.min()}")
print(f"Max bucket size           : {bucket_sizes_arr.max()}")
print(f"p90 bucket size           : {np.percentile(bucket_sizes_arr, 90):.2f}")
print(f"p99 bucket size           : {np.percentile(bucket_sizes_arr, 99):.2f}")

# A few useful counts
for thresh in [2, 5, 10, 50, 100, 500, 1000]:
    cnt = int(np.sum(bucket_sizes_arr >= thresh))
    print(f"Buckets with size >= {thresh:4d} : {cnt:,}")

# Show a few largest buckets
largest_items = sorted(part_map.items(), key=lambda kv: len(kv[1]), reverse=True)[:5]

print_memory_stats("After partition map build")

[4] Building partition map from weighted MinHash signatures...


Building band partitions: 100%|██████████| 187019/187019 [00:05<00:00, 31189.14poly/s]



Partition build time      : 9.13 sec
Total unique buckets      : 207,902
Total bucket assignments  : 9,350,950

=== Rebalance Settings ===
REB_MERGE_SMALL          : 0
REB_SPLIT_LARGE          : 3000

=== Rebalanced Bucket Stats ===
total_buckets            : 207902
mean_bucket_size         : 44.97768179238295
median_bucket_size       : 2.0
max_bucket_size          : 34095
p90_bucket_size          : 35.0
p99_bucket_size          : 1035.9899999999907

=== Bucket Size Diagnostics ===
Mean bucket size          : 44.98
Median bucket size        : 2.00
Min bucket size           : 1
Max bucket size           : 34095
p90 bucket size           : 35.00
p99 bucket size           : 1035.99
Buckets with size >=    2 : 109,873
Buckets with size >=    5 : 61,053
Buckets with size >=   10 : 41,351
Buckets with size >=   50 : 17,047
Buckets with size >=  100 : 11,501
Buckets with size >=  500 : 3,922
Buckets with size >= 1000 : 2,171
[After partition map build] RSS=18.90 GB | VMS=24.50 GB | Avail=38.

# Weighted Jaccard Helper for Reranking

This section defines the weighted Jaccard similarity used during the final reranking step.

In the BOI pipeline:
- weighted MinHash and banding generate candidates
- weighted Jaccard is then used to score and rerank those candidates exactly

This helper is part of the main retrieval pipeline.

In [10]:
# ================================
# Weighted Jaccard helper for reranking
# ================================

from numba import njit

@njit(cache=True)
def _weighted_jaccard_sparse_numba(ids_a, vals_a, ids_b, vals_b):
    if ids_a.size == 0 and ids_b.size == 0:
        return 1.0

    if ids_a.size == 0 or ids_b.size == 0:
        return 0.0

    i = 0
    j = 0
    num = 0.0
    den = 0.0

    while i < ids_a.size and j < ids_b.size:
        ida = ids_a[i]
        idb = ids_b[j]

        if ida == idb:
            va = vals_a[i]
            vb = vals_b[j]

            if va < vb:
                num += va
                den += vb
            else:
                num += vb
                den += va

            i += 1
            j += 1

        elif ida < idb:
            den += vals_a[i]
            i += 1

        else:
            den += vals_b[j]
            j += 1

    while i < ids_a.size:
        den += vals_a[i]
        i += 1

    while j < ids_b.size:
        den += vals_b[j]
        j += 1

    if den <= 0.0:
        return 1.0

    return num / den


def weighted_jaccard_from_tuples(enc_a, enc_b):
    ids_a, vals_a = enc_a
    ids_b, vals_b = enc_b
    return _weighted_jaccard_sparse_numba(ids_a, vals_a, ids_b, vals_b)

# Build Bucket Indexes

This section builds the bag of indexes (BOI) structure.

Design:
- only sufficiently large buckets get a local HNSW index
- smaller buckets are marked exact-only and searched directly at query time

Implementation detail:
- we first split buckets into large vs small
- then build indexes only for the large buckets

This avoids misleading progress estimates and keeps the multi-index stage practical.

In [11]:
# ================================
# Build bucket-local indexes in parallel (BOI core, PROCESSPOOL + reload)
# ================================

import os
import time
import shutil
import tempfile
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing.shared_memory import SharedMemory

import nmslib

# -------------------------
# Parallel build settings
# -------------------------
BUCKET_INDEX_MIN_SIZE = 1000
BUILD_WORKERS = 36
NUM_THREADS = 4

LOCAL_HNSW_M = 16
LOCAL_HNSW_EFC = 150
LOCAL_HNSW_EFS = 400

# Keep variable name for summary compatibility
BUCKET_INDEX_DIR = "IN_MEMORY_ONLY"

nested_L2 = 40
nested_R2 = 2

# Persistent index cache — keyed to dataset + HNSW config so changing
# any parameter automatically invalidates and rebuilds
_INDEX_CACHE_KEY = (
    f"L{L}_B{banding.B}_R{banding.R}"
    f"_M{LOCAL_HNSW_M}_efc{LOCAL_HNSW_EFC}"
    f"_minSz{BUCKET_INDEX_MIN_SIZE}"
    f"_rebSplit{REB_SPLIT_LARGE}"
    f"_{INDEX_TYPE}"
    f"_nL{nested_L2}_nR{nested_R2}"
    f"_sigvec"
)
BUCKET_INDEX_CACHE_DIR = os.path.join(
    "/mnt/data1/ruban/polygon-ann-multi-index/boi_cache",
    _dataset_key,
    f"bucket_indexes_{_INDEX_CACHE_KEY}"
)
print(f"BUCKET_INDEX_CACHE_DIR : {BUCKET_INDEX_CACHE_DIR}")

print("=== Parallel Bucket Index Build Parameters ===")
print(f"BUCKET_INDEX_MIN_SIZE  : {BUCKET_INDEX_MIN_SIZE}")
print(f"BUILD_WORKERS          : {BUILD_WORKERS}")
print(f"NUM_THREADS            : {NUM_THREADS}")
print(f"LOCAL_HNSW_M           : {LOCAL_HNSW_M}")
print(f"LOCAL_HNSW_EFC         : {LOCAL_HNSW_EFC}")
print(f"LOCAL_HNSW_EFS         : {LOCAL_HNSW_EFS}")
print(f"BUCKET_INDEX_DIR       : {BUCKET_INDEX_DIR}")
print(f"INDEX_TYPE             : {INDEX_TYPE}")


MAX_BUCKET_INDEX_SIZE = 2000

def split_buckets_by_size(part_map, min_bucket_size):
    """
    Split buckets into:
    - large buckets: build local HNSW index
    - small buckets: exact-only at query time
    """
    large = []
    small = []

    for bucket_key, members in part_map.items():
        member_ids = np.asarray(members, dtype=np.int32)
        size = int(member_ids.size)
        item = (bucket_key, member_ids, size)

        if size >= min_bucket_size:
            large.append(item)
        else:
            small.append(item)

    large.sort(key=lambda x: x[2], reverse=True)
    small.sort(key=lambda x: x[2], reverse=True)

    return large, small


# -------------------------
# Build-worker globals
# -------------------------
_BUCKET_BUILD_GLOBALS = {}


def _init_bucket_build_worker(shm_meta,
                               local_hnsw_m,
                               local_hnsw_efc,
                               local_hnsw_efs,
                               num_threads,
                               tmp_index_dir):
    from multiprocessing.shared_memory import SharedMemory
    global _BUCKET_BUILD_GLOBALS

    # Attach to shared sig_all — 100-dim WMH signatures, much smaller than x_dense
    meta = shm_meta["sig_all"]
    _shm_sig = SharedMemory(create=False, name=meta["name"])
    sig_all_shm = np.ndarray(meta["shape"], dtype=meta["dtype"], buffer=_shm_sig.buf)

    _BUCKET_BUILD_GLOBALS = {
        "sig_all": sig_all_shm,
        "_shm_sig": _shm_sig,
        "local_hnsw_m": local_hnsw_m,
        "local_hnsw_efc": local_hnsw_efc,
        "local_hnsw_efs": local_hnsw_efs,
        "num_threads": num_threads,
        "tmp_index_dir": tmp_index_dir,
        "nested_L2": nested_L2,
        "nested_R2": nested_R2,
    }

def _build_one_bucket_nested_lsh(args):
    """
    Build a nested LSH index for one large bucket.
    Applies a second round of MinHash banding on the bucket's signatures.
    Returns a dict of inner_band_key -> member_ids (as np.int32 arrays).
    """
    bucket_key, member_ids = args
    G = _BUCKET_BUILD_GLOBALS

    X_bucket = G["sig_all"][member_ids]   # shape: (bucket_size, 100)
    L2 = G["nested_L2"]
    R2 = G["nested_R2"]
    B2 = L2 // R2

    inner_map = {}
    for local_idx in range(len(member_ids)):
        sig = X_bucket[local_idx]
        for b in range(B2):
            band_key = tuple(sig[b * R2 : b * R2 + R2].tolist())
            k = (b, band_key)
            if k not in inner_map:
                inner_map[k] = []
            inner_map[k].append(local_idx)

    # Convert to numpy arrays
    inner_map_np = {
        k: np.asarray(v, dtype=np.int32)
        for k, v in inner_map.items()
    }

    return {
        "bucket_key": bucket_key,
        "member_ids": member_ids,
        "inner_map":  inner_map_np,
        "use_index":  True,
        "index_type": "nested_lsh",
        "worker_elapsed_sec": 0.0,
    }

def _build_one_bucket_index_worker(args):
    """
    Worker function:
    - receives one large bucket
    - builds local NMSLIB WeightedJaccard HNSW
    - saves it to a temp path
    - returns metadata only
    """
    import os
    import uuid
    import nmslib

    bucket_key, member_ids = args
    t0_worker = time.time()
    G = _BUCKET_BUILD_GLOBALS

    X_bucket = G["sig_all"][member_ids].astype(np.float32)

    idx = nmslib.init(space='WeightedJaccard', method='hnsw')
    idx.addDataPointBatch(X_bucket)

    idx.createIndex(
        {
            'M': G["local_hnsw_m"],
            'efConstruction': G["local_hnsw_efc"],
            'indexThreadQty': G["num_threads"],
            'post': 1
        },
        print_progress=False
    )
    idx.setQueryTimeParams({'efSearch': G["local_hnsw_efs"]})

    safe_name = f"bucket_{uuid.uuid4().hex}"
    index_path = os.path.join(G["tmp_index_dir"], safe_name)
    idx.saveIndex(index_path, save_data=True)

    # drop worker-local reference
    idx = None
    t_worker_done = time.time()

    return {
        "bucket_key": bucket_key,
        "member_ids": member_ids,
        "size": int(member_ids.size),
        "use_index": True,
        "index_path": index_path,
        "worker_elapsed_sec": float(t_worker_done - t0_worker),
    }


def _load_saved_weighted_index(index_path, efS):
    """
    Load one saved NMSLIB WeightedJaccard index into the parent process.
    """
    idx = nmslib.init(space='WeightedJaccard', method='hnsw')
    idx.loadIndex(index_path, load_data=True)
    idx.setQueryTimeParams({'efSearch': efS})
    return idx


def build_bucket_indexes_parallel(part_map,
                                  min_bucket_size=300,
                                  bucket_index_dir="IN_MEMORY_ONLY",
                                  build_workers=20,
                                  num_threads=8,
                                  local_hnsw_m=16,
                                  local_hnsw_efc=200,
                                  local_hnsw_efs=400):
    """
    Parallel BOI local-index builder (processpool version).

    Large buckets:
      - built in worker processes
      - saved to temp paths
      - reloaded in parent process as live indexes

    Small buckets:
      - stored as exact-only
      - searched directly at query time
    """
    t_build_0 = time.time()

    large_buckets, small_buckets = split_buckets_by_size(
        part_map=part_map,
        min_bucket_size=min_bucket_size
    )

    print("\n=== Bucket Split Summary ===")
    print(f"Total buckets             : {len(part_map):,}")
    print(f"Large buckets (indexed)   : {len(large_buckets):,}")
    print(f"Small buckets (exact-only): {len(small_buckets):,}")

    bucket_indexes = {}

    # Register small buckets first
    for bucket_key, member_ids, size in small_buckets:
        bucket_indexes[bucket_key] = {
            "member_ids": member_ids,
            "size": int(size),
            "use_index": False,
            "index": None,
            "index_path": None,
        }

    jobs = [(bucket_key, member_ids) for bucket_key, member_ids, _size in large_buckets]
    
    total_members_indexed = 0
    shm_root = "/dev/shm" if os.path.isdir("/dev/shm") else None
    tmp_index_dir = tempfile.mkdtemp(prefix="weighted_boi_hnsw_", dir=shm_root)

    try:
        ctx = mp.get_context("fork")

        # Select worker function based on INDEX_TYPE
        if INDEX_TYPE == "nested_lsh":
            _worker_fn = _build_one_bucket_nested_lsh
        else:
            _worker_fn = _build_one_bucket_index_worker

        # Build shared memory for sig_all so workers can access it
        _shm = SharedMemory(create=True, size=sig_all.nbytes)
        _shm_array = np.ndarray(sig_all.shape, dtype=sig_all.dtype, buffer=_shm.buf)
        _shm_array[:] = sig_all
        _inline_shm_meta = {
            "sig_all": {
                "name" : _shm.name,
                "shape": sig_all.shape,
                "dtype": sig_all.dtype,
            }
        }

        with ProcessPoolExecutor(
            max_workers=build_workers,
            mp_context=ctx,
            initializer=_init_bucket_build_worker,
            initargs=(
                _inline_shm_meta,
                local_hnsw_m,
                local_hnsw_efc,
                local_hnsw_efs,
                num_threads,
                tmp_index_dir,
            )
        ) as ex:
            futures = [ex.submit(_worker_fn, job) for job in jobs]

            worker_times = []
            reload_times = []

            for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Building local {INDEX_TYPE} indexes", unit="bucket"):
                res = fut.result()

                bucket_key = tuple(res["bucket_key"])
                member_ids = res["member_ids"]

                if INDEX_TYPE == "nested_lsh":
                    bucket_indexes[bucket_key] = {
                        "member_ids": member_ids,
                        "size":       int(member_ids.size),
                        "use_index":  True,
                        "index_type": "nested_lsh",
                        "inner_map":  res["inner_map"],
                        "index":      None,
                        "index_path": None,
                    }
                    worker_times.append(res["worker_elapsed_sec"])
                    reload_times.append(0.0)
                else:
                    t0_reload = time.time()
                    idx = _load_saved_weighted_index(res["index_path"], local_hnsw_efs)
                    reload_elapsed = time.time() - t0_reload

                    bucket_indexes[bucket_key] = {
                        "member_ids": member_ids,
                        "size":       int(res["size"]),
                        "use_index":  True,
                        "index_type": "hnsw",
                        "inner_map":  None,
                        "index":      idx,
                        "index_path": res["index_path"],
                    }
                    worker_times.append(res["worker_elapsed_sec"])
                    reload_times.append(reload_elapsed)

                total_members_indexed += int(member_ids.size)

    finally:
        shutil.rmtree(tmp_index_dir, ignore_errors=True)

    t_build = time.time() - t_build_0

    # ------------------------------------------------------------------
    # Build split_index: maps (band_idx, band_key, "SPLIT") -> sorted list
    # of full split-bucket keys. Built once here, used at every query to
    # replace the O(#all_buckets) scan in get_probe_bucket_keys().
    # ------------------------------------------------------------------
    split_index = {}
    for k in bucket_indexes:
        if len(k) == 4 and k[2] == "SPLIT":
            prefix = k[:3]                          # (band_idx, band_key, "SPLIT")
            if prefix not in split_index:
                split_index[prefix] = []
            split_index[prefix].append(k)
    for prefix in split_index:
        split_index[prefix].sort(key=lambda x: x[3])   # sort by sub-bucket index

    # ------------------------------------------------------------------
    # Precompute k_search per indexed bucket — removes the if/elif/else
    # branch from the hot query loop.
    # ------------------------------------------------------------------
    USE_HNSW_THRESHOLD = 1500
    for meta in bucket_indexes.values():
        if meta["use_index"]:
            sz = meta["size"]
            if sz < 200:
                meta["k_search"] = sz
            elif sz < 1000:
                meta["k_search"] = min(250, sz)
            else:
                meta["k_search"] = min(400, sz)
        else:
            meta["k_search"] = 0   # unused for exact buckets

    bucket_sizes = np.array([len(v) for v in part_map.values()], dtype=np.int32)

    build_stats = {
        "total_buckets": int(len(part_map)),
        "indexed_buckets": int(len(large_buckets)),
        "exact_only_buckets": int(len(small_buckets)),
        "total_members_indexed": int(total_members_indexed),
        "mean_bucket_size": float(bucket_sizes.mean()) if bucket_sizes.size > 0 else None,
        "median_bucket_size": float(np.median(bucket_sizes)) if bucket_sizes.size > 0 else None,
        "max_bucket_size": int(bucket_sizes.max()) if bucket_sizes.size > 0 else None,
        "build_time_sec": float(t_build),
        "bucket_index_dir": bucket_index_dir,
        "avg_worker_build_sec": float(np.mean(worker_times)) if len(worker_times) > 0 else None,
        "avg_reload_sec": float(np.mean(reload_times)) if len(reload_times) > 0 else None,
        "max_worker_build_sec": float(np.max(worker_times)) if len(worker_times) > 0 else None,
        "max_reload_sec": float(np.max(reload_times)) if len(reload_times) > 0 else None,
    }

    return bucket_indexes, split_index, build_stats


print("\n[BOI] Building bucket-local indexes in parallel...")
t0 = time.time()

BUCKET_INDEX_CACHE_DIR : /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/bucket_indexes_L100_B50_R2_M16_efc150_minSz1000_rebSplit3000_nested_lsh_nL40_nR2_sigvec
=== Parallel Bucket Index Build Parameters ===
BUCKET_INDEX_MIN_SIZE  : 1000
BUILD_WORKERS          : 36
NUM_THREADS            : 4
LOCAL_HNSW_M           : 16
LOCAL_HNSW_EFC         : 150
LOCAL_HNSW_EFS         : 400
BUCKET_INDEX_DIR       : IN_MEMORY_ONLY
INDEX_TYPE             : nested_lsh

[BOI] Building bucket-local indexes in parallel...


Your CPU supports instructions that this binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2
For maximum performance, you can install NMSLIB from sources 
pip install --no-binary :all: nmslib


In [12]:
import pickle, hashlib

def save_bucket_index_cache(bucket_indexes, split_index, build_stats, cache_dir):
    """
    Save all HNSW indexes + metadata to a persistent cache directory.
    Each indexed bucket's nmslib index is saved as a separate .bin file.
    All metadata (member_ids, k_search, split_index, stats) saved as one pickle.
    """
    os.makedirs(cache_dir, exist_ok=True)

    # Save each HNSW index to a named file
    meta_map = {}
    for bucket_key, entry in tqdm(bucket_indexes.items(), desc="Saving bucket indexes"):
        key_str = str(bucket_key)
        key_hash = hashlib.md5(key_str.encode()).hexdigest()

        record = {
            "member_ids": entry["member_ids"],
            "size":       entry["size"],
            "use_index":  entry["use_index"],
            "k_search":   entry["k_search"],
            "index_type": entry.get("index_type", "hnsw"),   # ADD THIS
            "index_path": None,
            "inner_map":  entry.get("inner_map", None),      # ADD THIS
        }

        if entry["use_index"] and entry["index"] is not None:
            idx_path = os.path.join(cache_dir, f"idx_{key_hash}.bin")
            entry["index"].saveIndex(idx_path, save_data=True)
            record["index_path"] = idx_path

        meta_map[bucket_key] = record

    # Save metadata + split_index + stats
    payload = {
        "meta_map":    meta_map,
        "split_index": split_index,
        "build_stats": build_stats,
    }
    with open(os.path.join(cache_dir, "payload.pkl"), "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved {len(meta_map):,} bucket entries to: {cache_dir}")


def load_bucket_index_cache(cache_dir, efs):
    """
    Load all HNSW indexes + metadata from a persistent cache directory.
    Returns bucket_indexes, split_index, build_stats — same as build function.
    """
    payload_path = os.path.join(cache_dir, "payload.pkl")
    if not os.path.exists(payload_path):
        return None, None, None

    print(f"Loading bucket index cache from: {cache_dir}")
    with open(payload_path, "rb") as f:
        payload = pickle.load(f)

    meta_map    = payload["meta_map"]
    split_index = payload["split_index"]
    build_stats = payload["build_stats"]

    bucket_indexes = {}
    n_indexed = sum(1 for v in meta_map.values() if v["use_index"])

    for bucket_key, record in tqdm(meta_map.items(), desc="Loading HNSW indexes"):
        idx = None
        if record["use_index"] and record["index_path"]:
            idx = nmslib.init(space='WeightedJaccard', method='hnsw')
            idx.loadIndex(record["index_path"], load_data=True)
            idx.setQueryTimeParams({'efSearch': efs})

        bucket_indexes[bucket_key] = {
            "member_ids": record["member_ids"],
            "size":       record["size"],
            "use_index":  record["use_index"],
            "index":      idx,
            "index_path": record["index_path"],
            "k_search":   record["k_search"],
        }

    print(f"Loaded {len(bucket_indexes):,} buckets "
          f"({n_indexed:,} indexed, "
          f"{len(bucket_indexes)-n_indexed:,} exact-only)")
    return bucket_indexes, split_index, build_stats


# -------------------------------------------------------
# Cache-aware build — load if available, build otherwise
# -------------------------------------------------------
t0 = time.time()

_payload_exists = os.path.exists(
    os.path.join(BUCKET_INDEX_CACHE_DIR, "payload.pkl")
)

if _payload_exists:
    print(f"[Cache hit] Loading bucket indexes from: {BUCKET_INDEX_CACHE_DIR}")
    bucket_indexes, split_index, bucket_index_build_stats = load_bucket_index_cache(
        cache_dir=BUCKET_INDEX_CACHE_DIR,
        efs=LOCAL_HNSW_EFS,
    )
else:
    print(f"[Cache miss] Building bucket indexes fresh...")
    bucket_indexes, split_index, bucket_index_build_stats = build_bucket_indexes_parallel(
        part_map=part_map,
        min_bucket_size=BUCKET_INDEX_MIN_SIZE,
        bucket_index_dir=BUCKET_INDEX_DIR,
        build_workers=BUILD_WORKERS,
        num_threads=NUM_THREADS,
        local_hnsw_m=LOCAL_HNSW_M,
        local_hnsw_efc=LOCAL_HNSW_EFC,
        local_hnsw_efs=LOCAL_HNSW_EFS,
    )
    print(f"[Cache save] Saving bucket indexes to: {BUCKET_INDEX_CACHE_DIR}")
    save_bucket_index_cache(
        bucket_indexes=bucket_indexes,
        split_index=split_index,
        build_stats=bucket_index_build_stats,
        cache_dir=BUCKET_INDEX_CACHE_DIR,
    )

t_bucket_build = time.time() - t0

print(f"\n=== Parallel Bucket Index Build Summary ===")
print(f"Total buckets             : {bucket_index_build_stats['total_buckets']:,}")
print(f"Indexed buckets           : {bucket_index_build_stats['indexed_buckets']:,}")
print(f"Exact-only buckets        : {bucket_index_build_stats['exact_only_buckets']:,}")
print(f"Total members indexed     : {bucket_index_build_stats['total_members_indexed']:,}")
print(f"Mean bucket size          : {bucket_index_build_stats['mean_bucket_size']:.2f}")
print(f"Median bucket size        : {bucket_index_build_stats['median_bucket_size']:.2f}")
print(f"Max bucket size           : {bucket_index_build_stats['max_bucket_size']:,}")
print(f"Build time                : {bucket_index_build_stats['build_time_sec']:.2f} sec ({bucket_index_build_stats['build_time_sec']/60:.2f} min)")
print(f"Index directory           : {bucket_index_build_stats['bucket_index_dir']}")
print(f"Avg worker build sec      : {bucket_index_build_stats['avg_worker_build_sec']:.4f}")
print(f"Avg reload sec            : {bucket_index_build_stats['avg_reload_sec']:.4f}")
print(f"Max worker build sec      : {bucket_index_build_stats['max_worker_build_sec']:.4f}")
print(f"Max reload sec            : {bucket_index_build_stats['max_reload_sec']:.4f}")

print_memory_stats("After parallel bucket-local index build")
import ctypes, gc
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)
print_memory_stats("After malloc_trim post index build")

[Cache miss] Building bucket indexes fresh...

=== Bucket Split Summary ===
Total buckets             : 207,902
Large buckets (indexed)   : 2,171
Small buckets (exact-only): 205,731


Building local nested_lsh indexes: 100%|██████████| 2171/2171 [00:02<00:00, 731.68bucket/s]


[Cache save] Saving bucket indexes to: /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/bucket_indexes_L100_B50_R2_M16_efc150_minSz1000_rebSplit3000_nested_lsh_nL40_nR2_sigvec


Saving bucket indexes: 100%|██████████| 207902/207902 [00:00<00:00, 501348.05it/s]


Saved 207,902 bucket entries to: /mnt/data1/ruban/polygon-ann-multi-index/boi_cache/pk-real0.002/bucket_indexes_L100_B50_R2_M16_efc150_minSz1000_rebSplit3000_nested_lsh_nL40_nR2_sigvec

=== Parallel Bucket Index Build Summary ===
Total buckets             : 207,902
Indexed buckets           : 2,171
Exact-only buckets        : 205,731
Total members indexed     : 5,149,371
Mean bucket size          : 44.98
Median bucket size        : 2.00
Max bucket size           : 34,095
Build time                : 9.63 sec (0.16 min)
Index directory           : IN_MEMORY_ONLY
Avg worker build sec      : 0.0000
Avg reload sec            : 0.0000
Max worker build sec      : 0.0000
Max reload sec            : 0.0000
[After parallel bucket-local index build] RSS=20.79 GB | VMS=25.57 GB | Avail=36.57/61.95 GB | Used=41.0%
[After malloc_trim post index build] RSS=20.73 GB | VMS=25.57 GB | Avail=36.63/61.95 GB | Used=40.9%


In [13]:
def get_probe_bucket_keys(bucket_indexes, split_index, band_idx, band_key, max_split_probes=2):
    """
    O(1) probe-key lookup.

    split_index: dict mapping (band_idx, band_key, "SPLIT") -> sorted list of full 4-tuple keys.
    Replaces the previous O(#all_buckets) full-dict scan.
    """
    keys = []

    exact_key = (band_idx, band_key)
    if exact_key in bucket_indexes:
        keys.append(exact_key)

    prefix = (band_idx, band_key, "SPLIT")
    split_keys = split_index.get(prefix)
    if split_keys:
        keys.extend(split_keys[:max_split_probes])

    return keys

In [21]:
# ================================
# Full BOI evaluation using fast query logic
# ================================

# Numba warmup
for _enc in encodings[:10]:
    pass

_warm_a = encodings[0]
_warm_b = encodings[1]
_ = weighted_jaccard_from_tuples(_warm_a, _warm_b)
print("Numba weighted Jaccard warmup done.")

# Full GT-backed query set
boi_eval_qids = [
    qid for qid in gt_qids
    if (gt_offsets[gt_qid_map[qid] + 1] - gt_offsets[gt_qid_map[qid]]) > 0
]

def compute_recall_at_k(pred_ids, gt_ids, k):
    if len(gt_ids) == 0:
        return None
    gt_k  = gt_ids[:k]
    denom = min(k, len(gt_k))
    if denom == 0:
        return None
    return len(set(pred_ids[:k]).intersection(gt_k)) / denom

import random
random.seed(42)
random.shuffle(boi_eval_qids)

print("boi_eval_qids size:", len(boi_eval_qids))
print("first 3:", boi_eval_qids[:3])
print("last 3:", boi_eval_qids[-3:])

# Fast-query parameters
FAST_TOPK = 500
FAST_ALPHA_MULT = 0.08
FAST_EPSILON = 12000
FAST_EFS = 800
FAST_HNSW_K_MULT = 1
QUERY_WORKERS = 36
CHUNKS_PER_WORKER = 8

Numba weighted Jaccard warmup done.
boi_eval_qids size: 44666
first 3: [np.int32(193004), np.int32(199603), np.int32(227154)]
last 3: [np.int32(188724), np.int32(194644), np.int32(230872)]


In [22]:
# ============================================================
# Cell 1 — GPU CSR Index Build
# Replaces: part_map, bucket_indexes, split_index
# ============================================================
import cupy as cp
import numpy as np
import time

# ---- Hash function: (band_idx, int64, int64) -> uint64 ----
# Polynomial hash — collision rate negligible at this scale
HASH_P1 = np.int64(2654435761)   # Knuth multiplicative
HASH_P2 = np.int64(805459861)
HASH_P3 = np.int64(3474701543)

def hash_band_key(band_idx, v0, v1):
    """Map (band_idx, v0, v1) -> uint64 scalar."""
    h = np.int64(band_idx) * HASH_P1
    h = h ^ (np.int64(v0) * HASH_P2)
    h = h ^ (np.int64(v1) * HASH_P3)
    return int(h & np.int64(0x7FFFFFFFFFFFFFFF))  # keep positive

# ---- Step 1: Assign integer IDs to all unique bucket keys ----
print("Step 1: Assigning bucket IDs...")
t0 = time.time()

bucket_key_to_id = {}   # (band_idx, v0, v1) -> int bucket_id
bucket_id_to_key = []   # bucket_id -> original key (for lookup)

for key in part_map.keys():
    band_idx = key[0]
    # Handle split buckets (4-tuple) and normal buckets (2-tuple)
    if len(key) == 2:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1)
    elif len(key) == 4:
        # SPLIT bucket: (band_idx, (v0,v1), 'SPLIT', j)
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1, key[3])  # include split index
    else:
        hk = key

    if hk not in bucket_key_to_id:
        bucket_key_to_id[hk] = len(bucket_id_to_key)
        bucket_id_to_key.append(key)

NUM_BUCKETS = len(bucket_key_to_id)
print(f"  Unique buckets: {NUM_BUCKETS:,}  ({time.time()-t0:.2f}s)")

# ---- Step 2: Build outer CSR (bucket -> member poly IDs) ----
print("Step 2: Building outer CSR...")
t0 = time.time()

# Count members per bucket
outer_counts = np.zeros(NUM_BUCKETS, dtype=np.int32)
for key, members in part_map.items():
    band_idx = key[0]
    if len(key) == 2:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1)
    elif len(key) == 4:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1, key[3])
    else:
        hk = key
    bid = bucket_key_to_id[hk]
    outer_counts[bid] = len(members)

# CSR row pointers
outer_offsets = np.zeros(NUM_BUCKETS + 1, dtype=np.int32)
outer_offsets[1:] = np.cumsum(outer_counts)
total_outer = int(outer_offsets[-1])

# CSR column indices (member poly IDs)
outer_members = np.empty(total_outer, dtype=np.int32)
for key, members in part_map.items():
    band_idx = key[0]
    if len(key) == 2:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1)
    elif len(key) == 4:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1, key[3])
    else:
        hk = key
    bid = bucket_key_to_id[hk]
    s = outer_offsets[bid]
    e = outer_offsets[bid + 1]
    outer_members[s:e] = np.asarray(members, dtype=np.int32)

print(f"  Outer CSR: {NUM_BUCKETS:,} buckets, {total_outer:,} assignments ({time.time()-t0:.2f}s)")

# ---- Step 3: Build per-bucket metadata arrays ----
print("Step 3: Building bucket metadata...")
t0 = time.time()

# is_large[bid] = 1 if this bucket has a nested LSH inner map
is_large    = np.zeros(NUM_BUCKETS, dtype=np.int32)
bucket_size = outer_counts.copy()

# Map original bucket_indexes keys -> bucket IDs
orig_key_to_bid = {}
for key in bucket_indexes.keys():
    band_idx = key[0]
    if len(key) == 2:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1)
    elif len(key) == 4:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1, key[3])
    else:
        hk = key
    if hk in bucket_key_to_id:
        orig_key_to_bid[key] = bucket_key_to_id[hk]

for key, meta in bucket_indexes.items():
    if meta['use_index']:  # removed index_type check — cache has None
        bid = orig_key_to_bid.get(key)
        if bid is not None:
            is_large[bid] = 1

n_large = int(is_large.sum())
n_small = NUM_BUCKETS - n_large
print(f"  Large buckets (nested LSH): {n_large:,}")
print(f"  Small buckets (exact scan): {n_small:,}")
print(f"  Metadata built ({time.time()-t0:.2f}s)")

# ---- Step 4: Build inner CSR for large buckets ----
print("Step 4: Building inner CSR for large buckets...")
t0 = time.time()

# inner_key_to_id: (bid, inner_band_idx, iv0, iv1) -> inner_bucket_id
inner_key_to_id   = {}
inner_bucket_list = []  # inner_bucket_id -> (bid, inner_band_idx, iv0, iv1)

# First pass: collect all unique inner bucket keys
for key, meta in bucket_indexes.items():
    if not meta['use_index']:
        continue
    bid = orig_key_to_bid.get(key)
    if bid is None:
        continue
    for (ib, ikey), local_ids in meta['inner_map'].items():
        iv0, iv1 = int(ikey[0]), int(ikey[1])
        ik = (bid, ib, iv0, iv1)
        if ik not in inner_key_to_id:
            inner_key_to_id[ik] = len(inner_bucket_list)
            inner_bucket_list.append(ik)

NUM_INNER_BUCKETS = len(inner_key_to_id)

# Inner CSR: inner_bucket -> local member indices (local to their outer bucket)
inner_counts = np.zeros(NUM_INNER_BUCKETS, dtype=np.int32)
for key, meta in bucket_indexes.items():
    if not meta['use_index']:
        continue
    bid = orig_key_to_bid.get(key)
    if bid is None:
        continue
    for (ib, ikey), local_ids in meta['inner_map'].items():
        iv0, iv1 = int(ikey[0]), int(ikey[1])
        ik = (bid, ib, iv0, iv1)
        iid = inner_key_to_id[ik]
        inner_counts[iid] = len(local_ids)

inner_offsets_np = np.zeros(NUM_INNER_BUCKETS + 1, dtype=np.int32)
inner_offsets_np[1:] = np.cumsum(inner_counts)
total_inner = int(inner_offsets_np[-1])

inner_members_np = np.empty(total_inner, dtype=np.int32)
for key, meta in bucket_indexes.items():
    if not meta['use_index']:
        continue
    bid = orig_key_to_bid.get(key)
    if bid is None:
        continue
    for (ib, ikey), local_ids in meta['inner_map'].items():
        iv0, iv1 = int(ikey[0]), int(ikey[1])
        ik = (bid, ib, iv0, iv1)
        iid = inner_key_to_id[ik]
        s = inner_offsets_np[iid]
        e = inner_offsets_np[iid + 1]
        inner_members_np[s:e] = local_ids

print(f"  Inner CSR: {NUM_INNER_BUCKETS:,} inner buckets, {total_inner:,} assignments ({time.time()-t0:.2f}s)")

# ---- Step 5: Build band_key -> bucket_id lookup table ----
print("Step 5: Building query-time band lookup table...")
t0 = time.time()

# For each (band_idx, v0, v1) -> bucket_id
# Store as sorted arrays for binary search on GPU
# Format: for each band b, store sorted list of (hash_val, bucket_id)

# We build B separate lookup arrays (one per band)
# Each: sorted by hash(v0, v1), stores bucket_id
band_lookup = []   # band_lookup[b] = (hash_vals_sorted, bucket_ids_sorted)

for b in range(B):
    hvals = []
    bids  = []
    for key in part_map.keys():
        if key[0] != b:
            continue
        if len(key) == 2:
            v0, v1 = int(key[1][0]), int(key[1][1])
            hk = (b, v0, v1)
        elif len(key) == 4:
            v0, v1 = int(key[1][0]), int(key[1][1])
            hk = (b, v0, v1, key[3])
        else:
            continue
        h = hash_band_key(b, v0, v1)
        hvals.append(h)
        bids.append(bucket_key_to_id[hk])

    if hvals:
        hvals = np.array(hvals, dtype=np.int64)
        bids  = np.array(bids,  dtype=np.int32)
        order = np.argsort(hvals)
        band_lookup.append((hvals[order], bids[order]))
    else:
        band_lookup.append((np.array([], dtype=np.int64),
                            np.array([], dtype=np.int32)))

print(f"  Band lookup built for {B} bands ({time.time()-t0:.2f}s)")

# ---- Step 6: Move everything to GPU ----
print("Step 6: Moving index to GPU...")
t0 = time.time()

gpu_outer_offsets = cp.asarray(outer_offsets)   # (NUM_BUCKETS+1,) int32
gpu_outer_members = cp.asarray(outer_members)   # (total_outer,)   int32
gpu_is_large      = cp.asarray(is_large)         # (NUM_BUCKETS,)   int32
gpu_bucket_size   = cp.asarray(bucket_size)      # (NUM_BUCKETS,)   int32
gpu_inner_offsets = cp.asarray(inner_offsets_np) # (NUM_INNER+1,)   int32
gpu_inner_members = cp.asarray(inner_members_np) # (total_inner,)   int32

# Band lookup arrays on GPU
gpu_band_hvals = [cp.asarray(band_lookup[b][0]) for b in range(B)]
gpu_band_bids  = [cp.asarray(band_lookup[b][1]) for b in range(B)]

# sig_all on GPU (int64 for exact band key matching)
gpu_sig_all = cp.asarray(sig_all)   # (50000, 100) int64

# inner bucket lookup: (bid, ib) -> range in inner CSR
# Build a flat array: for each large bucket bid,
# store offset into a per-bid inner bucket list
# We need: given (bid, ib, iv0, iv1) -> inner_bucket_id
# Store as: for each bid, sorted (hash(ib,iv0,iv1), inner_id)
large_bid_list = np.where(is_large)[0]
# For each large bucket, store its inner bucket hash+id arrays
inner_per_bid_hvals  = {}
inner_per_bid_iids   = {}

for key, meta in bucket_indexes.items():
    if not meta['use_index']:
        continue
    bid = orig_key_to_bid.get(key)
    if bid is None:
        continue
    hvals_b, iids_b = [], []
    for (ib, ikey), local_ids in meta['inner_map'].items():
        iv0, iv1 = int(ikey[0]), int(ikey[1])
        ik  = (bid, ib, iv0, iv1)
        iid = inner_key_to_id[ik]
        h   = hash_band_key(ib, iv0, iv1)
        hvals_b.append(h)
        iids_b.append(iid)
    hvals_b = np.array(hvals_b, dtype=np.int64)
    iids_b  = np.array(iids_b,  dtype=np.int32)
    order   = np.argsort(hvals_b)
    inner_per_bid_hvals[bid] = cp.asarray(hvals_b[order])
    inner_per_bid_iids[bid]  = cp.asarray(iids_b[order])

print(f"  GPU transfer done ({time.time()-t0:.2f}s)")

# ---- Summary ----
vram_used = (
    gpu_outer_offsets.nbytes + gpu_outer_members.nbytes +
    gpu_is_large.nbytes + gpu_inner_offsets.nbytes +
    gpu_inner_members.nbytes + gpu_sig_all.nbytes +
    sum(v.nbytes for v in gpu_band_hvals) +
    sum(v.nbytes for v in gpu_band_bids)
) / 1e6

print(f"\n=== GPU CSR Index Built ===")
print(f"Outer CSR   : {NUM_BUCKETS:,} buckets, {total_outer:,} assignments")
print(f"Inner CSR   : {NUM_INNER_BUCKETS:,} inner buckets, {total_inner:,} assignments")
print(f"Large buckets: {n_large:,} | Small: {n_small:,}")
print(f"VRAM used   : {vram_used:.1f} MB")
print(f"VRAM free   : {cp.cuda.Device(0).mem_info[0]/1e6:.1f} MB")

Step 1: Assigning bucket IDs...
  Unique buckets: 207,902  (0.12s)
Step 2: Building outer CSR...
  Outer CSR: 207,902 buckets, 9,350,950 assignments (0.31s)
Step 3: Building bucket metadata...
  Large buckets (nested LSH): 2,171
  Small buckets (exact scan): 205,731
  Metadata built (0.18s)
Step 4: Building inner CSR for large buckets...
  Inner CSR: 1,951,126 inner buckets, 102,987,420 assignments (3.26s)
Step 5: Building query-time band lookup table...


/tmp/ipykernel_2654390/15222236.py:18: RuntimeWarning: overflow encountered in scalar multiply
  h = h ^ (np.int64(v0) * HASH_P2)
/tmp/ipykernel_2654390/15222236.py:19: RuntimeWarning: overflow encountered in scalar multiply
  h = h ^ (np.int64(v1) * HASH_P3)


  Band lookup built for 50 bands (0.98s)
Step 6: Moving index to GPU...
  GPU transfer done (4.37s)

=== GPU CSR Index Built ===
Outer CSR   : 207,902 buckets, 9,350,950 assignments
Inner CSR   : 1,951,126 inner buckets, 102,987,420 assignments
Large buckets: 2,171 | Small: 205,731
VRAM used   : 648.3 MB
VRAM free   : 22607.8 MB


In [23]:
# ---- Move encoding arrays to GPU ----
print("Moving encoding arrays to GPU...")
gpu_ids_flat  = cp.asarray(ids_flat)
gpu_vals_flat = cp.asarray(vals_flat)
gpu_offsets   = cp.asarray(offsets)
print(f"  ids_flat  : {gpu_ids_flat.nbytes/1e9:.2f} GB")
print(f"  vals_flat : {gpu_vals_flat.nbytes/1e9:.2f} GB")
print(f"  VRAM free : {cp.cuda.Device(0).mem_info[0]/1e9:.2f} GB")

Moving encoding arrays to GPU...
  ids_flat  : 4.59 GB
  vals_flat : 4.59 GB
  VRAM free : 18.02 GB


In [24]:
# ============================================================
# Cell 3 — Full GPU Pipeline
# CPU: band key lookup → bucket IDs (2ms, no hashing needed)
# GPU: inner LSH probe + scoreboard + prune + batch rerank
# ============================================================

import cupyx

# ---- Precompute per-large-bucket: band_idx it belongs to ----
# So at query time we know which signature slice to use
print("Precomputing large bucket metadata...")

# For each large bucket (by bid), store:
#   - which outer band_idx it came from
#   - its member_ids (for global ID resolution)
#   - nested_L2, nested_R2 for inner banding

large_bid_to_bandidx = {}   # bid -> outer band_idx
large_bid_to_members = {}   # bid -> np.int32 member_ids (local index -> global poly_id)

for key, meta in bucket_indexes.items():
    if not meta['use_index']:
        continue
    bid = orig_key_to_bid.get(key)
    if bid is None:
        continue
    large_bid_to_bandidx[bid] = key[0]   # outer band_idx
    large_bid_to_members[bid] = meta['member_ids']  # shape (bucket_size,) int32

print(f"Large buckets precomputed: {len(large_bid_to_bandidx)}")


# ---- Build inner bucket lookup: (bid, inner_hash) -> inner_bucket_id ----
# For GPU binary search: per large bucket, sorted (inner_hash, inner_id) arrays
# Already built as inner_per_bid_hvals / inner_per_bid_iids
# Flatten into one big array with a bid-level offset table

print("Building flat inner lookup...")
t0 = time.time()

large_bids_sorted = sorted(large_bid_to_bandidx.keys())
n_large_bids      = len(large_bids_sorted)

# bid -> position in large_bids_sorted
large_bid_to_pos  = {bid: pos for pos, bid in enumerate(large_bids_sorted)}

# Per large-bucket inner lookup sizes
inner_lookup_sizes   = np.array([
    len(inner_per_bid_hvals[bid]) for bid in large_bids_sorted
], dtype=np.int32)

inner_lookup_offsets = np.zeros(n_large_bids + 1, dtype=np.int32)
inner_lookup_offsets[1:] = np.cumsum(inner_lookup_sizes)
total_inner_lookup   = int(inner_lookup_offsets[-1])

# Flat inner hash + inner_id arrays
flat_inner_hvals = np.empty(total_inner_lookup, dtype=np.int64)
flat_inner_iids  = np.empty(total_inner_lookup, dtype=np.int32)

for pos, bid in enumerate(large_bids_sorted):
    s = inner_lookup_offsets[pos]
    e = inner_lookup_offsets[pos + 1]
    flat_inner_hvals[s:e] = inner_per_bid_hvals[bid].get()
    flat_inner_iids[s:e]  = inner_per_bid_iids[bid].get()

# Member ID resolution: for each large bucket, map local_idx -> global poly_id
# Flatten all member arrays
member_array_sizes   = np.array([
    len(large_bid_to_members[bid]) for bid in large_bids_sorted
], dtype=np.int32)
member_array_offsets = np.zeros(n_large_bids + 1, dtype=np.int32)
member_array_offsets[1:] = np.cumsum(member_array_sizes)
total_members_flat   = int(member_array_offsets[-1])

flat_members = np.empty(total_members_flat, dtype=np.int32)
for pos, bid in enumerate(large_bids_sorted):
    s = member_array_offsets[pos]
    e = member_array_offsets[pos + 1]
    flat_members[s:e] = large_bid_to_members[bid]

# Move to GPU
gpu_flat_inner_hvals    = cp.asarray(flat_inner_hvals)
gpu_flat_inner_iids     = cp.asarray(flat_inner_iids)
gpu_inner_lookup_offsets= cp.asarray(inner_lookup_offsets)
gpu_flat_members        = cp.asarray(flat_members)
gpu_member_offsets      = cp.asarray(member_array_offsets)

print(f"  Flat inner lookup : {total_inner_lookup:,} entries")
print(f"  Flat members      : {total_members_flat:,} entries")
print(f"  Done ({time.time()-t0:.2f}s)")
print(f"  VRAM free         : {cp.cuda.Device(0).mem_info[0]/1e9:.2f} GB")

# ---- GPU kernel: inner LSH probe for large buckets ----
_INNER_LSH_KERNEL = cp.RawKernel(r'''
__device__ long long hash_inner(int ib, long long v0, long long v1) {
    long long h = (long long)ib * 2654435761LL;
    h = h ^ (v0 * 805459861LL);
    h = h ^ (v1 * 3474701543LL);
    return h & 0x7FFFFFFFFFFFFFFFLL;
}

__device__ int bsearch64(
    const long long* arr, int n, long long target
) {
    int lo = 0, hi = n - 1;
    while (lo <= hi) {
        int mid = (lo + hi) >> 1;
        if (arr[mid] == target) return mid;
        if (arr[mid] < target)  lo = mid + 1;
        else                    hi = mid - 1;
    }
    return -1;
}

extern "C" __global__
void gpu_inner_lsh_probe(
    // Query info
    const long long* __restrict__ sig_all,      // (N_total, L) int64
    const int*       __restrict__ query_ids,    // (Q,)
    const int Q,
    const int L,

    // Which large buckets to probe per query:
    // For query qi, probe large_bucket_ids[qb_offsets[qi]..qb_offsets[qi+1]]
    const int* __restrict__ large_bucket_ids,   // flat list of (pos in large_bids_sorted)
    const int* __restrict__ qb_offsets,         // (Q+1,)

    // Inner lookup (per large bucket, sorted by inner hash)
    const long long* __restrict__ flat_inner_hvals,
    const int*       __restrict__ flat_inner_iids,
    const int*       __restrict__ inner_lookup_offsets, // (n_large+1,)

    // Inner CSR (inner_bucket_id -> local member indices)
    const int* __restrict__ inner_offsets,      // (NUM_INNER+1,)
    const int* __restrict__ inner_members,      // (total_inner,)

    // Member resolution (pos -> global poly IDs)
    const int* __restrict__ flat_members,       // flat member arrays
    const int* __restrict__ member_offsets,     // (n_large+1,)

    // Nested LSH params
    const int nested_B2,
    const int nested_R2,

    // Scoreboard (Q, DATA_END)
    float* __restrict__ scoreboard,
    const int DATA_END
) {
    int qi  = blockIdx.x;
    if (qi >= Q) return;

    int qid = query_ids[qi];
    int tid = threadIdx.x;

    int n_probe = qb_offsets[qi + 1] - qb_offsets[qi];
    if (n_probe == 0) return;

    // Each thread handles one large bucket
    for (int pi = tid; pi < n_probe; pi += blockDim.x) {
        int pos = large_bucket_ids[qb_offsets[qi] + pi];

        // Inner lookup range for this bucket
        int il_start = inner_lookup_offsets[pos];
        int il_size  = inner_lookup_offsets[pos + 1] - il_start;

        // Member offset for this bucket
        int mem_start = member_offsets[pos];

        // Probe all nested_B2 inner bands
        for (int ib = 0; ib < nested_B2; ib++) {
            // Get inner band key from query signature
            int sig_base = qid * L + ib * nested_R2;
            long long iv0 = sig_all[sig_base];
            long long iv1 = sig_all[sig_base + 1];

            long long ih = hash_inner(ib, iv0, iv1);

            // Binary search in this bucket's inner lookup
            const long long* il_hvals = flat_inner_hvals + il_start;
            const int*       il_iids  = flat_inner_iids  + il_start;

            int ipos = bsearch64(il_hvals, il_size, ih);
            if (ipos < 0) continue;

            int inner_id = il_iids[ipos];

            // Get local member indices from inner CSR
            int ms = inner_offsets[inner_id];
            int me = inner_offsets[inner_id + 1];

            // Resolve local -> global and scatter-add to scoreboard
            for (int m = ms; m < me; m++) {
                int local_idx = inner_members[m];
                int global_id = flat_members[mem_start + local_idx];
                if (global_id < DATA_END) {
                    atomicAdd(&scoreboard[qi * DATA_END + global_id], 1.0f);
                }
            }
        }
    }
}
''', 'gpu_inner_lsh_probe')

print("Inner LSH kernel compiled.")

_SMALL_PROBE_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void gpu_small_probe(
    const int* __restrict__ sb_starts,   // (n_pairs,) start in outer_members
    const int* __restrict__ sb_sizes,    // (n_pairs,) bucket size
    const int* __restrict__ sb_qidxs,   // (n_pairs,) which query
    const int* __restrict__ outer_members,
    float*     __restrict__ scoreboard,
    const int  DATA_END,
    const int  n_pairs
) {
    // One block per (query, bucket) pair
    int pair_idx = blockIdx.x;
    if (pair_idx >= n_pairs) return;

    int qi      = sb_qidxs[pair_idx];
    int ms      = sb_starts[pair_idx];
    int bk_size = sb_sizes[pair_idx];

    // Threads within block share the work for this bucket
    for (int t = threadIdx.x; t < bk_size; t += blockDim.x) {
        int poly_id = outer_members[ms + t];
        if (poly_id < DATA_END) {
            atomicAdd(&scoreboard[qi * DATA_END + poly_id], 1.0f);
        }
    }
}
''', 'gpu_small_probe')

print("Small probe kernel compiled.")


# ---- CPU band key lookup (fast — replaces hashing) ----
def cpu_get_bucket_ids_for_query(query_sig):
    small_bids = []
    large_poss = []

    # Access sig as raw numpy — avoid repeated attribute lookups
    sig_np = query_sig if isinstance(query_sig, np.ndarray) else np.asarray(query_sig)

    for b in range(min(B, 40)):
        # Inline band key extraction — faster than banding.get_band()
        s = b * R
        band_key  = (int(sig_np[s]), int(sig_np[s + 1]))
        exact_key = (b, band_key)

        bid = orig_key_to_bid.get(exact_key)
        if bid is not None:
            if is_large[bid]:
                pos = large_bid_to_pos.get(bid)
                if pos is not None:
                    large_poss.append(pos)
            else:
                small_bids.append(bid)

        # Split buckets
        split_prefix = (b, band_key, 'SPLIT')
        sl = split_index.get(split_prefix, [])
        for sk in sl:
            bid = orig_key_to_bid.get(sk)
            if bid is not None:
                if is_large[bid]:
                    pos = large_bid_to_pos.get(bid)
                    if pos is not None:
                        large_poss.append(pos)
                else:
                    small_bids.append(bid)

    return small_bids, large_poss

# ================================
# Kernel 3: Batch Weighted Jaccard rerank
# ================================
_WJ_BATCH_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void batch_wj_all_queries(
    const int*   __restrict__ query_idx,
    const int*   __restrict__ cand_ids,
    const float* __restrict__ scores,
    const int*   __restrict__ q_starts,
    const int*   __restrict__ q_nnzs,
    const int*   __restrict__ all_ids,
    const float* __restrict__ all_vals,
    const long*  __restrict__ all_offsets,
    float*       __restrict__ out_scores,
    const int    total_pairs
) {
    int ci = blockIdx.x * blockDim.x + threadIdx.x;
    if (ci >= total_pairs) return;
    int qi      = query_idx[ci];
    int poly_id = cand_ids[ci];
    int q_start = q_starts[qi];
    int q_nnz   = q_nnzs[qi];
    const int*   q_ids  = all_ids  + q_start;
    const float* q_vals = all_vals + q_start;
    int db_start = (int)all_offsets[poly_id];
    int db_end   = (int)all_offsets[poly_id + 1];
    int db_nnz   = db_end - db_start;
    const int*   db_ids  = all_ids  + db_start;
    const float* db_vals = all_vals + db_start;
    float num = 0.0f, den = 0.0f;
    int i = 0, j = 0;
    while (i < q_nnz && j < db_nnz) {
        int ia = q_ids[i], ib = db_ids[j];
        if (ia == ib) {
            float va = q_vals[i], vb = db_vals[j];
            num += (va < vb) ? va : vb;
            den += (va > vb) ? va : vb;
            i++; j++;
        } else if (ia < ib) { den += q_vals[i]; i++;
        } else               { den += db_vals[j]; j++; }
    }
    while (i < q_nnz)  { den += q_vals[i];  i++; }
    while (j < db_nnz) { den += db_vals[j]; j++; }
    out_scores[ci] = (den <= 0.0f) ? 1.0f : num / den;
}
''', 'batch_wj_all_queries')


def gpu_rerank_batch(query_ids, probe_results, topK=500):
    """Batch Weighted Jaccard rerank — all queries in one kernel launch."""
    if len(query_ids) == 0:
        return {}

    Q = len(query_ids)
    cand_arrays  = [probe_results.get(qid, np.empty(0, dtype=np.int32))
                    for qid in query_ids]
    cand_lengths = np.array([len(c) for c in cand_arrays], dtype=np.int32)
    q_offsets_np = np.zeros(Q + 1, dtype=np.int32)
    q_offsets_np[1:] = np.cumsum(cand_lengths)
    total_pairs  = int(cand_lengths.sum())

    if total_pairs == 0:
        return {qid: [] for qid in query_ids}

    cand_flat_np = np.concatenate(cand_arrays).astype(np.int32)
    query_idx_np = np.repeat(np.arange(Q, dtype=np.int32), cand_lengths)
    q_starts     = np.array([int(offsets[qid])     for qid in query_ids], dtype=np.int32)
    q_nnzs       = (np.array([int(offsets[qid+1]) for qid in query_ids], dtype=np.int32) - q_starts)

    query_idx_gpu = cp.asarray(query_idx_np)
    cand_gpu      = cp.asarray(cand_flat_np)
    q_starts_gpu  = cp.asarray(q_starts)
    q_nnzs_gpu    = cp.asarray(q_nnzs)
    scores_gpu    = cp.empty(total_pairs, dtype=cp.float32)

    BLOCK = 256
    GRID  = (total_pairs + BLOCK - 1) // BLOCK
    _WJ_BATCH_KERNEL(
        (GRID,), (BLOCK,),
        (query_idx_gpu, cand_gpu, scores_gpu,
         q_starts_gpu, q_nnzs_gpu,
         gpu_ids_flat, gpu_vals_flat, gpu_offsets,
         scores_gpu, total_pairs)
    )
    cp.cuda.Stream.null.synchronize()

    scores_cpu = scores_gpu.get()
    cands_cpu  = cand_gpu.get()
    results    = {}
    for qi, qid in enumerate(query_ids):
        s, e = int(q_offsets_np[qi]), int(q_offsets_np[qi+1])
        if s == e:
            results[qid] = []
            continue
        q_scores = scores_cpu[s:e]
        q_cands  = cands_cpu[s:e]
        if len(q_scores) > topK:
            top_idx  = np.argpartition(q_scores, -topK)[-topK:]
            q_scores = q_scores[top_idx]
            q_cands  = q_cands[top_idx]
        order = np.argsort(-q_scores)
        results[qid] = list(zip(q_cands[order].tolist(), q_scores[order].tolist()))
    return results

print("Batch rerank kernel compiled.")


# ---- Full GPU batch query pipeline ----
def gpu_query_pipeline(query_ids_list,
                       topK=500,
                       alpha_mult=0.08,
                       epsilon=4000,
                       data_end=DATA_END,
                       nested_B2=nested_L2 // nested_R2,
                       nested_R2_=nested_R2,
                       show_progress=False):
    """
    Full GPU pipeline:
    1. CPU band lookup  → bucket IDs per query (fast, exact, no hashing)
    2. GPU small probe  → scoreboard scatter-add
    3. GPU large probe  → inner LSH + scoreboard scatter-add
    4. CPU prune        → top-epsilon candidates
    5. GPU batch rerank → weighted Jaccard top-K
    """
    Q = len(query_ids_list)
    alpha = B * alpha_mult

    # ---- Step 1: CPU band lookup for all queries ----
    t_lookup = time.time()

    all_small_bids  = []   # flat
    all_large_poss  = []   # flat
    q_small_offsets = [0]  # per query
    q_large_offsets = [0]  # per query

    for qid in query_ids_list:
        query_sig = sig_all[qid]
        small_bids, large_poss = cpu_get_bucket_ids_for_query(query_sig)
        all_small_bids.extend(small_bids)
        all_large_poss.extend(large_poss)
        q_small_offsets.append(len(all_small_bids))
        q_large_offsets.append(len(all_large_poss))

    t_lookup = time.time() - t_lookup

    # ---- Step 2 ----
    scoreboard_gpu = cp.zeros((Q, data_end), dtype=cp.float32)

    # ---- Step 3: GPU small bucket probe via kernel (numpy-vectorized build) ----
    t_small = time.time()
    if all_small_bids:
        # Vectorized build — no Python loops over postings
        small_bids_np  = np.array(all_small_bids, dtype=np.int32)  # (total_small_buckets,)
        q_small_off_np = np.array(q_small_offsets, dtype=np.int32)  # (Q+1,)

        # For each small bucket, get its start+size from outer_offsets
        sb_starts_np = outer_offsets[small_bids_np]               # (n_pairs,)
        sb_sizes_np  = outer_offsets[small_bids_np + 1] - sb_starts_np  # (n_pairs,)

        # Build qidx: for each bucket position, which query owns it
        # q_small_off_np[qi]..q_small_off_np[qi+1] = bucket positions for query qi
        counts   = np.diff(q_small_off_np)                        # (Q,) buckets per query
        sb_qidxs_np = np.repeat(np.arange(Q, dtype=np.int32), counts)  # (n_pairs,)

        n_pairs = len(small_bids_np)

        gpu_sb_starts = cp.asarray(sb_starts_np.astype(np.int32))
        gpu_sb_sizes  = cp.asarray(sb_sizes_np.astype(np.int32))
        gpu_sb_qidxs  = cp.asarray(sb_qidxs_np)

        _SMALL_PROBE_KERNEL(
            (n_pairs,), (128,),
            (
                gpu_sb_starts,
                gpu_sb_sizes,
                gpu_sb_qidxs,
                gpu_outer_members,
                scoreboard_gpu,
                data_end,
                n_pairs,
            )
        )
        cp.cuda.Stream.null.synchronize()

    t_small = time.time() - t_small

    # ---- Step 4: GPU large bucket probe (inner LSH kernel) ----
    t_large = time.time()
    if all_large_poss:
        large_poss_gpu   = cp.asarray(np.array(all_large_poss,  dtype=np.int32))
        q_large_off_gpu  = cp.asarray(np.array(q_large_offsets, dtype=np.int32))
        qids_gpu         = cp.asarray(np.array(query_ids_list,  dtype=np.int32))

        THREADS = min(64, max(1, max(
            q_large_offsets[qi+1] - q_large_offsets[qi]
            for qi in range(Q)
        )))

        _INNER_LSH_KERNEL(
            (Q,), (THREADS,),
            (
                gpu_sig_all,
                qids_gpu,
                Q, L,
                large_poss_gpu,
                q_large_off_gpu,
                gpu_flat_inner_hvals,
                gpu_flat_inner_iids,
                gpu_inner_lookup_offsets,
                gpu_inner_offsets,
                gpu_inner_members,
                gpu_flat_members,
                gpu_member_offsets,
                nested_B2, nested_R2_,
                scoreboard_gpu,
                data_end,
            )
        )
        cp.cuda.Stream.null.synchronize()

    t_large = time.time() - t_large

    # ---- Step 5: CPU prune — fully vectorized ----
    t_prune = time.time()
    scoreboard_cpu = scoreboard_gpu.get()   # (Q, DATA_END)

    # Alpha threshold: zero out below-alpha entries
    if alpha > 0:
        scoreboard_cpu[scoreboard_cpu < alpha] = 0.0

    probe_results = {}

    # Check if any query needs epsilon cap
    touched_counts = np.count_nonzero(scoreboard_cpu, axis=1)  # (Q,)
    needs_cap      = touched_counts > epsilon

    for qi, qid in enumerate(query_ids_list):
        if touched_counts[qi] == 0:
            probe_results[qid] = np.empty(0, dtype=np.int32)
            continue

        touched = np.where(scoreboard_cpu[qi] > 0)[0].astype(np.int32)

        if needs_cap[qi]:
            scores  = scoreboard_cpu[qi][touched]
            top_idx = np.argpartition(scores, -epsilon)[-epsilon:]
            touched = touched[top_idx]

        probe_results[qid] = touched

    t_prune = time.time() - t_prune

    # ---- Step 6: GPU batch rerank ----
    t_rerank = time.time()
    results  = gpu_rerank_batch(query_ids_list, probe_results, topK=topK)
    t_rerank = time.time() - t_rerank

    t_total = t_lookup + t_small + t_large + t_prune + t_rerank

    timing = {
        "lookup_ms" : t_lookup * 1000,
        "small_ms"  : t_small  * 1000,
        "large_ms"  : t_large  * 1000,
        "prune_ms"  : t_prune  * 1000,
        "rerank_ms" : t_rerank * 1000,
        "total_ms"  : t_total  * 1000,
        "qps"       : Q / t_total,
    }

    return results, probe_results, timing


# ---- Warmup ----
print("Warming up full pipeline...")
_ = gpu_query_pipeline(
    boi_eval_qids[:2],
    topK=FAST_TOPK,
    alpha_mult=FAST_ALPHA_MULT,
    epsilon=FAST_EPSILON,
    show_progress=False,
)
cp.cuda.Stream.null.synchronize()
print("Warmup done.")

# ---- Smoke test: 10 queries ----
print("\nSmoke test: 10 queries...")
_test_qids = boi_eval_qids[:10]
_results, _probe, _timing = gpu_query_pipeline(
    _test_qids,
    topK=FAST_TOPK,
    alpha_mult=FAST_ALPHA_MULT,
    epsilon=FAST_EPSILON,
)

print(f"\n── Timing breakdown ──────────────────")
print(f"CPU lookup  : {_timing['lookup_ms']:.2f}ms")
print(f"GPU small   : {_timing['small_ms']:.2f}ms")
print(f"GPU large   : {_timing['large_ms']:.2f}ms")
print(f"CPU prune   : {_timing['prune_ms']:.2f}ms")
print(f"GPU rerank  : {_timing['rerank_ms']:.2f}ms")
print(f"Total       : {_timing['total_ms']:.2f}ms")
print(f"QPS         : {_timing['qps']:.1f}")

print(f"\n── Candidate counts ──────────────────")
for qid in _test_qids:
    print(f"  qid={qid}  candidates={len(_probe[qid])}")

# Quick recall check
rec10 = []
for qid in _test_qids:
    pred = [p for p, _ in _results[qid]]
    gi   = gt_qid_map[qid]
    gt   = gt_neighbors[gt_offsets[gi]:gt_offsets[gi+1]]
    r    = compute_recall_at_k(pred, gt, 10)
    if r is not None:
        rec10.append(r)
print(f"\nMean Recall@10 : {np.mean(rec10):.4f}  (baseline 0.9949)")

Precomputing large bucket metadata...
Large buckets precomputed: 2171
Building flat inner lookup...
  Flat inner lookup : 1,951,126 entries
  Flat members      : 5,149,371 entries
  Done (0.17s)
  VRAM free         : 18.02 GB
Inner LSH kernel compiled.
Small probe kernel compiled.
Batch rerank kernel compiled.
Warming up full pipeline...
Warmup done.

Smoke test: 10 queries...

── Timing breakdown ──────────────────
CPU lookup  : 8.63ms
GPU small   : 0.99ms
GPU large   : 35.72ms
CPU prune   : 12.99ms
GPU rerank  : 36.31ms
Total       : 94.63ms
QPS         : 105.7

── Candidate counts ──────────────────
  qid=193004  candidates=12000
  qid=199603  candidates=12000
  qid=227154  candidates=12000
  qid=190379  candidates=12000
  qid=220874  candidates=12000
  qid=189027  candidates=5550
  qid=188124  candidates=12000
  qid=232642  candidates=12000
  qid=232264  candidates=12000
  qid=190514  candidates=12000

Mean Recall@10 : 0.9800  (baseline 0.9949)


In [25]:
# ================================
# GPU Kernel Warmup
# ================================
print("Warming up GPU kernels...")
_ = gpu_query_pipeline(
    boi_eval_qids[:5],
    topK=FAST_TOPK,
    alpha_mult=FAST_ALPHA_MULT,
    epsilon=FAST_EPSILON,
)
cp.cuda.Stream.null.synchronize()
print("Warmup done.")

Warming up GPU kernels...
Warmup done.


In [26]:
# ================================
# Full Evaluation
# ================================
# Process in batches to avoid OOM on scoreboard allocation
EVAL_BATCH_SIZE = 200  # tune down if still OOM

results_full = {}
probe_full   = {}
timing_accum = {"lookup_ms":0,"small_ms":0,"large_ms":0,
                "prune_ms":0,"rerank_ms":0,"total_ms":0}

t_wall = time.time()
for i in tqdm(range(0, len(boi_eval_qids), EVAL_BATCH_SIZE), desc="Evaluating"):
    batch = boi_eval_qids[i:i+EVAL_BATCH_SIZE]
    r, p, t = gpu_query_pipeline(
        batch,
        topK=FAST_TOPK,
        alpha_mult=FAST_ALPHA_MULT,
        epsilon=FAST_EPSILON,
    )
    results_full.update(r)
    probe_full.update(p)
    for k in timing_accum:
        timing_accum[k] += t[k]
    cp.cuda.Stream.null.synchronize()
    cp.get_default_memory_pool().free_all_blocks()

t_wall_total = time.time() - t_wall
timing_full  = timing_accum
timing_full["qps"] = len(boi_eval_qids) / t_wall_total
timing_full["total_ms"] = t_wall_total * 1000

rec10, rec50, rec500 = [], [], []
for qid in boi_eval_qids:
    if qid not in results_full: continue
    pred = [p for p, _ in results_full[qid]]
    gi   = gt_qid_map[qid]
    gt   = gt_neighbors[gt_offsets[gi]:gt_offsets[gi+1]]
    r10  = compute_recall_at_k(pred, gt, 10)
    r50  = compute_recall_at_k(pred, gt, 50)
    r500 = compute_recall_at_k(pred, gt, 500)
    if r10  is not None: rec10.append(r10)
    if r50  is not None: rec50.append(r50)
    if r500 is not None: rec500.append(r500)

print(f"\n=== FINAL GPU PIPELINE RESULTS ===")
print(f"Queries   : {len(boi_eval_qids):,}")
print(f"QPS       : {timing_full['qps']:.1f}   (baseline 323)")
print(f"Speedup   : {timing_full['qps']/323:.2f}x")
print(f"\nR@10  : {np.mean(rec10):.4f}  (baseline 0.9949)")
print(f"R@50  : {np.mean(rec50):.4f}  (baseline 0.9964)")
print(f"R@500 : {np.mean(rec500):.4f}  (baseline 0.9916)")

total_ms = timing_full['total_ms']
print(f"\n── Timing breakdown ──────────────────")
for key, label in [("lookup_ms","CPU lookup"), ("small_ms","GPU small"),
                   ("large_ms","GPU large"), ("prune_ms","CPU prune"),
                   ("rerank_ms","GPU rerank")]:
    ms = timing_full[key]
    print(f"  {label:12s} : {ms:7.0f}ms  ({ms/total_ms*100:.1f}%)")
print(f"  {'Total':12s} : {total_ms:7.0f}ms")
print_memory_stats("After evaluation")

Evaluating: 100%|██████████| 224/224 [04:44<00:00,  1.27s/it]



=== FINAL GPU PIPELINE RESULTS ===
Queries   : 44,666
QPS       : 157.2   (baseline 323)
Speedup   : 0.49x

R@10  : 0.9922  (baseline 0.9949)
R@50  : 0.9946  (baseline 0.9964)
R@500 : 0.9937  (baseline 0.9916)

── Timing breakdown ──────────────────
  CPU lookup   :   29108ms  (10.2%)
  GPU small    :    4437ms  (1.6%)
  GPU large    :   24785ms  (8.7%)
  CPU prune    :   66961ms  (23.6%)
  GPU rerank   :  158323ms  (55.7%)
  Total        :  284103ms
[After evaluation] RSS=43.10 GB | VMS=67.09 GB | Avail=14.21/61.95 GB | Used=77.1%
